# ML-Enhanced Value Strategy — Main Backtest Pipeline

## What this notebook does

This is the central pipeline of the project. It takes the two cleaned datasets produced by the data notebooks (`kelly_final.csv` for 1957–2021 and `features_2022_2024.csv` for 2022–2024), runs four XGBoost models and two benchmarks through a full out-of-sample backtest with a complete cost model, and produces the allocation CSVs that feed every chart and table in the presentation.

The notebook is structured in two parts:

**Part 1 — The Pipeline (Steps 1–23)**: data loading, preprocessing, model training, OOS and in-sample backtests, FF5 benchmark, output CSVs, performance reporting, and plots. Run cells 1–18 in order to reproduce all results from scratch.

**Part 2 — Post-Run Analysis**: standalone analysis cells added after the main pipeline was complete — AUM capacity analysis, IVE/VTV ETF benchmark comparison, and short book diagnostics. These cells read from the saved output CSVs and can be run independently.

## The four models

| Model | Signals | Count | Rationale |
|---|---|---|---|
| A — Value Only | VALUE | 7 | Baseline: what can ML do with valuation alone? |
| B — Value+Quality | VALUE + QUALITY | 24 | QARP: avoid value traps with quality overlay |
| C — Val+Quality+Mom | VALUE + QUALITY + MOMENTUM | 30 | Full model |
| D — Val+Momentum | VALUE + MOMENTUM | 13 | Best risk-adjusted: momentum confirms cheapness |

## Train / OOS split — the most important design decision

The model is trained on data up to and including December 2014. The out-of-sample period is January 2015 to November 2024. The model never sees a single OOS month during training. The 2022–2024 extension adds 36 months that did not exist when Kelly et al. published — these are the cleanest out-of-sample observations in the entire project.

## Cost model

All reported returns are net of four cost layers applied in sequence:
1. Transaction costs: 50bps one-way on turnover
2. Stock borrow fee: 150bps/yr on the short book
3. Short rebate: RF minus 25bps haircut (floored at zero)
4. Capital efficiency: divide by 1.5 (Reg T 50% margin requirement)

---
# PART 1 — THE PIPELINE
---

## Steps 1–3 — Configuration, data loading, and preprocessing

### Global configuration

All constants that control the backtest are defined here in one place. Changing a cost assumption or a date boundary here propagates everywhere automatically — nothing is hardcoded inside functions.

The coverage filter controls how we handle models with quality signals. Because quality signals have 43–66% NaN rates in Kelly, models B and C require a minimum fraction of quality features to be non-null before a stock is admitted to the universe. Without this filter, the model trains on a feature matrix where most quality entries are median-imputed zeros, which causes it to downweight quality signals entirely. The filter ensures the model only trains on rows where quality data is genuinely informative.

The turnover buffer keeps incumbent stocks in the long and short books if they only slip by one or two deciles — reducing unnecessary turnover and transaction costs for models B and C.

### Data loading

Kelly is loaded in chunks to avoid loading all 3.9M rows into memory at once. We only load the columns we actually need — the 30 signals plus permno, date, and ret. The extension dataset is small enough to load directly.

### Coverage filter and rank normalisation

The coverage filter runs first, then rank normalisation. The normalisation maps every feature to \[-1, +1\] within each month's cross-section. Missing values are filled with the cross-sectional median (rank = 0) before normalisation — unknown quality is treated as average, not penalised. This matches Kelly et al.'s exact preprocessing.

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
import statsmodels.api as sm
import matplotlib.pyplot as plt
import random
import os
import gc
import warnings
warnings.filterwarnings('ignore')

# Fix all random seeds for reproducibility
# GPU runs may still produce slightly different results due to floating point order
random.seed(42)
np.random.seed(42)
os.environ['PYTHONHASHSEED'] = '42'

# ============================================================
# FEATURE SETS
# ============================================================

VALUE    = ['bm','bm_ia','cfp','cfp_ia','ep','sp','dy']

QUALITY  = ['roaq','roeq','gma','operprof',
            'acc','absacc','pctacc',
            'tb','cashdebt','stdacc',
            'agr','egr','lgr','invest',
            'lev','ms','nincr']

MOMENTUM = ['mom1m','mom6m','mom12m','mom36m','chmom','indmom']

FEATURE_SETS = {
    'A_Value':           VALUE,
    'B_ValueQuality':    VALUE + QUALITY,
    'C_ValueQualityMom': VALUE + QUALITY + MOMENTUM,
    'D_ValMOM':          VALUE + MOMENTUM,
}

TARGET = 'ret'

# Coverage filter thresholds — B and C require quality data to be present
# A and D have no quality signals so the filter is irrelevant (0.0 = off)
COVERAGE = {
    'A_Value':           0.0,
    'B_ValueQuality':    0.5,
    'C_ValueQualityMom': 0.6,
    'D_ValMOM':          0.0,
}

# Turnover buffer: incumbents within N deciles of the top/bottom are kept
# Applied to B and C only — reduces unnecessary churn without changing the core selection
TURNOVER_BUFFER = {
    'A_Value':           0,
    'B_ValueQuality':    2,
    'C_ValueQualityMom': 2,
    'D_ValMOM':          0,
}

# Cost constants — all four layers of the cost model
TC_ONEWAY            = 0.0050   # 50bps one-way transaction cost
BORROW_COST_ANNUAL   = 0.0150   # 150bps/yr stock borrow fee on short book
REBATE_HAIRCUT       = 0.0025   # 25bps/yr prime broker haircut on RF rebate
CAPITAL_EFFICIENCY   = 1 / 1.5  # Reg T: need $1.50 capital per $1 L/S

# Train / OOS / Extension split
# TRAIN_END is the last month the model ever sees during training
# Everything after TEST_START is strictly out-of-sample
TRAIN_END  = '2014-12-01'
TEST_START = '2015-01-01'
TEST_END   = '2021-12-01'
EXT_START  = '2022-01-01'
EXT_END    = '2024-11-01'

# ============================================================
# STEP 1: LOAD DATA
# ============================================================

# Load Kelly in chunks — at 3.9M rows we only want the columns we need
# Loading all 94 Kelly columns would use ~4GB RAM unnecessarily
print("Loading Kelly dataset...")
all_features = VALUE + QUALITY + MOMENTUM
load_cols    = ['permno','date',TARGET] + all_features

chunks = []
for chunk in pd.read_csv('kelly_final.csv',
                          usecols=lambda c: c in load_cols,
                          chunksize=500_000,
                          parse_dates=['date']):
    chunks.append(chunk)
kelly = pd.concat(chunks, ignore_index=True)
kelly = kelly.sort_values(['permno','date']).reset_index(drop=True)
del chunks
gc.collect()
print(f"Kelly: {len(kelly):,} rows  |  {kelly['date'].min()} → {kelly['date'].max()}")

print("\nLoading extension dataset...")
ext = pd.read_csv('features_2022_2024.csv', parse_dates=['date'])
ext = ext.sort_values(['permno','date']).reset_index(drop=True)
print(f"Extension: {len(ext):,} rows  |  {ext['date'].min()} → {ext['date'].max()}")

# ============================================================
# STEP 2: COVERAGE FILTER
# ============================================================

def apply_coverage_filter(df, feature_cols, min_coverage=0.0):
    if min_coverage <= 0:
        return df
    # Only count quality signals for the coverage check
    # Value and momentum signals are not affected by the NaN problem
    quality_cols = [c for c in feature_cols if c not in VALUE + MOMENTUM]
    if not quality_cols:
        return df
    coverage = df[quality_cols].notna().mean(axis=1)
    df_out   = df[coverage >= min_coverage].copy()
    pct      = 1 - len(df_out) / len(df)
    print(f"  Coverage >={min_coverage:.0%}: {len(df):,} → {len(df_out):,} ({pct:.1%} dropped)")
    return df_out

# ============================================================
# STEP 3: RANK NORMALISATION
# ============================================================

def rank_normalize(df, feature_cols):
    df = df.copy()

    def cs_rank(x):
        # Cross-sectional rank normalised to [-1, +1]
        # rank=0 means median stock — this is also the imputed value for NaN
        r = x.rank(method='average')
        if len(r) <= 1:
            return r * 0
        return (r - 1) / (len(r) - 1) * 2 - 1

    # Fill NaN with cross-sectional median before ranking
    # Missing feature = average stock, not penalised or rewarded
    df[feature_cols] = df.groupby('date')[feature_cols].transform(
        lambda x: x.fillna(x.median())
    )
    df[feature_cols] = df[feature_cols].fillna(0)
    df[feature_cols] = df.groupby('date')[feature_cols].transform(cs_rank)
    return df

## Steps 4–5 — FF5 factors and Piotroski F-score

### FF5 factors

We download the Fama-French 5-factor data directly from Ken French's website. This gives us the monthly risk-free rate (RF), market excess return (MKT), and the four other factors (SMB, HML, RMW, CMA) needed for alpha regressions. The file is a fixed-width CSV with a non-standard header that requires careful parsing — we find the first line that starts with a 6-digit date to locate the data start.

### Piotroski F-score

The Piotroski F-score (0–9) is computed every month for every stock in the portfolio using nine binary signals derived from our existing quality features. It is stored in the output as `avg_fscore_long` and `avg_fscore_short` for diagnostic purposes — allowing us to verify that the long book consistently has better financial health than the short book.

The hard filter (keeping only long stocks with F-score ≥ 5) is deliberately set to `False` for all models in the main backtest. The ML model already learns to weight quality signals nonlinearly — adding a hard binary filter on top would introduce an arbitrary threshold and double-count the quality information.

In [ ]:
# ============================================================
# STEP 4: FF5 FACTORS
# ============================================================

def get_ff5():
    import requests, zipfile, io
    url = ("https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/"
           "ftp/F-F_Research_Data_5_Factors_2x3_CSV.zip")
    print("Downloading FF5 factors...")
    r = requests.get(url)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    with z.open(z.namelist()[0]) as f:
        raw = f.read().decode('utf-8')
    lines = raw.split('\n')
    # Find the first line that starts with a 6-digit date (e.g. '192607')
    # The file has a multi-line header before the data starts
    start = 0
    for i, line in enumerate(lines):
        s = line.strip()
        if s and s[0].isdigit() and len(s.split(',')[0].strip()) == 6:
            start = i
            break
    from io import StringIO
    ff5 = pd.read_csv(
        StringIO('\n'.join(lines[start:])),
        header=None,
        names=['date','MKT','SMB','HML','RMW','CMA','RF'],
        on_bad_lines='skip'
    )
    ff5 = ff5[ff5['date'].astype(str).str.strip().str.len() == 6]
    ff5['date'] = pd.to_datetime(ff5['date'].astype(str).str.strip(), format='%Y%m')
    ff5 = ff5.set_index('date')
    ff5 = ff5.apply(pd.to_numeric, errors='coerce') / 100
    ff5 = ff5.dropna()
    print(f"FF5: {ff5.index[0]} → {ff5.index[-1]}")
    return ff5

ff5       = get_ff5()
rf_series = ff5['RF']

# ============================================================
# STEP 5: PIOTROSKI F-SCORE (diagnostic only)
# ============================================================

def compute_fscore(month_df):
    # Nine binary signals — each scores 1 if the condition is met, 0 otherwise
    # Uses our existing quality features rather than recomputing from raw accounting
    # Stored in output for diagnostics: long book should consistently outscore short book
    df    = month_df.copy()
    score = pd.Series(0, index=df.index)
    for col in ['roaq','cfp','nincr']:
        if col in df.columns:
            score += (df[col] > 0).astype(int)
    if 'acc'      in df.columns: score += (df['acc']      < 0).astype(int)
    if 'lev'      in df.columns: score += (df['lev']      < 0).astype(int)
    if 'cashdebt' in df.columns: score += (df['cashdebt'] > 0).astype(int)
    if 'egr'      in df.columns: score += (df['egr']      < 0).astype(int)
    for col in ['gma','operprof']:
        if col in df.columns:
            score += (df[col] > 0).astype(int)
    return score

## Step 6 — Portfolio builder with full cost model

This function is the core of the pipeline. It runs every month for every model and produces a single row of portfolio results.

### What it does

Given predicted returns for all stocks in the universe, it assigns deciles using `pd.qcut`, identifies the top and bottom decile as the long and short books, applies the turnover buffer for incumbents, and computes net returns after all four cost layers.

### The decile label fix — why we use actual_max / actual_min

The original code hardcoded `decile == 9` for the long book and `decile == 0` for the short book. When prediction variance is low (Models A and B have prediction standard deviation of ~0.006 vs ~0.011 for C), `pd.qcut` occasionally forms 9 instead of 10 deciles due to tied predictions. Hardcoding 9 then silently missed the long book entirely, causing 162 missing months for Model A. Using `actual_max` and `actual_min` — the actual top and bottom labels produced by `pd.qcut` that month — fixed this completely.

### The cost calculation walkthrough

The costs are applied in this exact order:
1. Long book net return = gross return minus TC on turnover
2. Short book P&L = minus gross short return plus rebate minus borrow minus TC on turnover
3. L/S pre-capital-efficiency = long net + short P&L
4. Final net return = pre-capeff × (1/1.5)

The rebate is floored at zero — in near-zero rate environments (2015–2019) it contributes nothing. In 2022–2024 with RF around 4.5–5%, it contributes approximately 4.25–4.75% per year, partially offsetting the borrow cost.

In [ ]:
# ============================================================
# STEP 6: PORTFOLIO BUILDER — FULL COST MODEL
# ============================================================

def build_portfolio_with_costs(month_df, n_deciles, date,
                                prev_long, prev_short,
                                tc               = TC_ONEWAY,
                                rf_rate          = 0.0,
                                borrow_cost_ann  = BORROW_COST_ANNUAL,
                                rebate_haircut   = REBATE_HAIRCUT,
                                cap_efficiency   = True,
                                turnover_buffer  = 0.0,
                                piotroski_filter = False,
                                piotroski_min    = 5,
                                piotroski_min_n  = 20):
    try:
        month_df = month_df.copy()
        month_df['decile'] = pd.qcut(
            month_df['pred'], n_deciles,
            labels=False, duplicates='drop'
        )
    except ValueError:
        return None

    # Use actual_max and actual_min — NOT hardcoded 9 and 0
    # When prediction variance is low, qcut may form fewer than n_deciles
    # Hardcoding 9 caused 162 missing months for Model A before this fix
    actual_max = month_df['decile'].max()
    actual_min = month_df['decile'].min()
    if actual_max == actual_min or pd.isna(actual_max):
        return None

    long_df  = month_df[month_df['decile'] == actual_max].copy()
    short_df = month_df[month_df['decile'] == actual_min].copy()

    # Turnover buffer: keep incumbents that are still within N deciles of the top/bottom
    # Reduces unnecessary churn without changing which stocks are in the core universe
    if turnover_buffer > 0 and len(prev_long) > 0:
        carry_long = month_df[
            (month_df['permno'].isin(prev_long)) &
            (month_df['decile'] >= actual_max - turnover_buffer)
        ]
        long_df = pd.concat([long_df, carry_long]).drop_duplicates('permno')
        carry_short = month_df[
            (month_df['permno'].isin(prev_short)) &
            (month_df['decile'] <= actual_min + turnover_buffer)
        ]
        short_df = pd.concat([short_df, carry_short]).drop_duplicates('permno')

    # F-score is always computed for diagnostics even when the hard filter is off
    month_df['fscore'] = compute_fscore(month_df)
    long_df  = long_df.merge(month_df[['permno','fscore']], on='permno', how='left')
    short_df = short_df.merge(month_df[['permno','fscore']], on='permno', how='left')
    avg_fscore_long  = long_df['fscore'].mean()
    avg_fscore_short = short_df['fscore'].mean()

    # Piotroski hard filter — currently always False
    # The ML model already incorporates quality signals nonlinearly
    # A hard binary filter would double-count quality and impose an arbitrary threshold
    if piotroski_filter:
        filtered = long_df[long_df['fscore'] >= piotroski_min]
        if len(filtered) >= piotroski_min_n:
            long_df = filtered

    if len(long_df) == 0:
        return None

    # Gross returns — equal-weighted within each book
    long_ret_gross  = long_df[TARGET].mean()
    short_ret_gross = short_df[TARGET].mean() if len(short_df) > 0 else 0.0

    curr_long  = set(long_df['permno'].values)
    curr_short = set(short_df['permno'].values)

    # Turnover = fraction of new names entering the book this month
    # First month always 100% since there is no previous portfolio
    long_to  = len(curr_long  - prev_long)  / len(curr_long)  if prev_long  else 1.0
    short_to = len(curr_short - prev_short) / len(curr_short) if prev_short else 1.0

    # Monthly cost rates
    borrow_monthly  = borrow_cost_ann / 12
    haircut_monthly = rebate_haircut  / 12
    # Short rebate: receive RF on cash proceeds from short sales, minus broker haircut
    # Floored at zero — never pay more than we receive
    net_rebate = max(rf_rate - haircut_monthly, 0.0)

    # Layer 1 and 2: long net return and short book P&L
    long_ret_net = long_ret_gross - long_to * tc
    short_pnl    = (-short_ret_gross
                    + net_rebate
                    - borrow_monthly
                    - short_to * tc)

    # Layer 3: combined L/S before capital efficiency adjustment
    ls_ret_net_pre = long_ret_net + short_pnl

    # Layer 4: capital efficiency — Reg T requires $1.50 to run $1 L/S
    ls_ret_net = ls_ret_net_pre * CAPITAL_EFFICIENCY if cap_efficiency \
                 else ls_ret_net_pre

    # Zero-cost reference — useful for understanding what costs actually take away
    ls_ret_gross = long_ret_gross - short_ret_gross

    tc_cost  = (long_to + short_to) * tc
    net_fin  = net_rebate - borrow_monthly

    return {
        'date':                  date,
        'long_ret':              long_ret_net,
        'short_pnl':             short_pnl,
        'ls_ret':                ls_ret_net,
        'ls_ret_gross':          ls_ret_gross,
        'ls_ret_pre_capeff':     ls_ret_net_pre,
        'long_turnover':         long_to,
        'short_turnover':        short_to,
        'avg_turnover':          (long_to + short_to) / 2,
        'n_long':                len(long_df),
        'n_short':               len(short_df),
        'mkt_ret':               month_df[TARGET].mean(),
        'avg_fscore_long':       avg_fscore_long,
        'avg_fscore_short':      avg_fscore_short,
        'tc_cost_monthly':       tc_cost,
        'borrow_cost_monthly':   borrow_monthly,
        'rebate_monthly':        net_rebate,
        'net_financing_monthly': net_fin,
        'rf_rate':               rf_rate,
        'long_permnos':          curr_long,
        'short_permnos':         curr_short,
    }

## Steps 7–8 — Training helpers and GPU detection

### Training helpers

`prepare_training_data` handles the preprocessing needed before XGBoost training: drop rows with missing target, fill NaN features with the column median, clip at ±10 to prevent extreme rank values from dominating gradients, and remove any remaining non-finite values. This runs every annual retrain.

`predict_month` applies the same preprocessing to a single month's cross-section and returns predicted return scores. These scores are not calibrated return predictions — they are ordinal ranking scores. What matters is the order, not the level.

`make_xgb` constructs a fresh XGBoost model with fixed hyperparameters. Parameters are fixed at literature conventions from Kelly et al. and are not optimised on the OOS period — any optimisation using OOS data would constitute data snooping.

### GPU detection

The pipeline tries to use CUDA (NVIDIA GPU) first. If unavailable it falls back to CPU. On the full Kelly dataset (3.9M rows, 500 estimators), GPU training takes approximately 8 minutes per retrain vs 45 minutes on CPU. The annual retraining over 10 OOS years means GPU saves roughly 6 hours per model run.

In [ ]:
# ============================================================
# STEP 7: TRAINING HELPERS
# ============================================================

def prepare_training_data(train_df, feature_cols, target_col='ret'):
    train       = train_df.dropna(subset=[target_col]).copy()
    X           = train[feature_cols].copy()
    y           = train[target_col].values
    # Fill NaN with column median, then zero, then clip extreme values
    # The clip at ±10 prevents extreme rank normalised values from dominating
    col_medians = X.median()
    X           = X.fillna(col_medians).fillna(0).clip(-10, 10)
    X_vals      = X.values
    # Final safety check: drop any remaining non-finite rows
    mask        = np.isfinite(X_vals).all(axis=1) & np.isfinite(y)
    return X_vals[mask], y[mask]

def predict_month(month_df, feature_cols, model):
    # Apply identical preprocessing as training before predicting
    # The output is a ranking score, not a calibrated return estimate
    X = (month_df[feature_cols]
         .fillna(month_df[feature_cols].median())
         .fillna(0)
         .clip(-10, 10)
         .values)
    return model.predict(X)

def make_xgb(device):
    # Hyperparameters fixed at literature conventions — not optimised on OOS data
    # min_child_weight=20 ensures each leaf represents at least 20 stocks
    # subsample and colsample_bytree introduce stochasticity to prevent overfitting
    return XGBRegressor(
        n_estimators     = 500,
        learning_rate    = 0.01,
        max_depth        = 6,
        min_child_weight = 20,
        subsample        = 0.5,
        colsample_bytree = 0.5,
        reg_alpha        = 0.1,
        reg_lambda       = 0.1,
        random_state     = 42,
        seed             = 42,
        verbosity        = 0,
        device           = device,
        tree_method      = 'hist',
    )

# ============================================================
# STEP 8: GPU CHECK
# ============================================================

# Try CUDA first — GPU training is ~5x faster on the full Kelly dataset
# Falls back to CPU silently if CUDA is not available
try:
    _t = XGBRegressor(device='cuda', n_estimators=10, verbosity=0)
    _t.fit([[1,2],[3,4]], [1,0])
    DEVICE = 'cuda'
    print("CUDA confirmed — using GPU")
except Exception:
    DEVICE = 'cpu'
    print("No CUDA — using CPU")

## Step 9 — OOS backtest engine

This is the function that produces the results. It runs the expanding-window backtest for a single model across the full OOS period (2015–2024).

### The expanding window design

The model retrains once per calendar year using all data strictly before the current test year. The mandatory pre-train on data up to December 2014 guarantees a valid model for January 2015 — without this, the first OOS month would fail if the annual retrain threshold was not met.

The key invariant: at any test date T, the model has only ever seen data before T. No OOS month is ever included in any training run.

### The extension period

After the Kelly OOS loop (2015–2021), a final model is trained on the complete Kelly dataset (1957–2021) and used for all 2022–2024 extension months. This represents the model a real investor would deploy starting January 2022 after observing the full available history.

In [ ]:
# ============================================================
# STEP 9: OOS BACKTEST ENGINE
# ============================================================

def run_backtest(kelly_df, ext_df, feature_cols, model_name,
                 train_end        = TRAIN_END,
                 test_start       = TEST_START,
                 test_end         = TEST_END,
                 ext_start        = EXT_START,
                 ext_end          = EXT_END,
                 n_deciles        = 10,
                 transaction_cost = TC_ONEWAY,
                 min_coverage     = 0.0,
                 turnover_buffer  = 0.0,
                 piotroski_filter = False):

    print(f"\n{'='*60}")
    print(f"OOS — {model_name} | {len(feature_cols)} features")
    print(f"Coverage: {min_coverage:.0%} | Buffer: {turnover_buffer}")
    print(f"Train: 1957–{train_end} | OOS: {test_start}–{test_end} | Ext: {ext_start}–{ext_end}")
    print(f"{'='*60}")

    kelly_f    = apply_coverage_filter(kelly_df, feature_cols, min_coverage)
    ext_f      = apply_coverage_filter(ext_df,   feature_cols, min_coverage)
    kelly_proc = rank_normalize(kelly_f, feature_cols)
    ext_proc   = rank_normalize(ext_f,   feature_cols)

    test_dates = sorted(kelly_proc[
        (kelly_proc['date'] >= test_start) &
        (kelly_proc['date'] <= test_end)
    ]['date'].unique())

    ext_dates = sorted(ext_proc[
        (ext_proc['date'] >= ext_start) &
        (ext_proc['date'] <= ext_end)
    ]['date'].unique())

    results           = []
    model             = None
    last_retrain_year = None
    prev_long         = set()
    prev_short        = set()

    # Mandatory pre-train: guarantees a valid model for the first OOS month
    # Without this, January 2015 would have no model if the retrain threshold failed
    print(f"\nPre-training on data ≤ {train_end}...")
    pretrain     = kelly_proc[kelly_proc['date'] <= train_end]
    X_pre, y_pre = prepare_training_data(pretrain, feature_cols)
    if len(y_pre) < 1000:
        raise ValueError(f"Only {len(y_pre)} obs in pre-train — check dates")
    model = make_xgb(DEVICE)
    model.fit(X_pre, y_pre)
    last_retrain_year = pd.Timestamp(test_start).year - 1
    print(f"  Pre-trained on {len(y_pre):,} obs")

    # OOS loop: retrain once per calendar year, predict monthly
    print(f"Running OOS ({len(test_dates)} months)...")
    for test_date in test_dates:
        current_year = test_date.year

        if current_year != last_retrain_year:
            train      = kelly_proc[kelly_proc['date'] < test_date]
            X_tr, y_tr = prepare_training_data(train, feature_cols)
            if len(y_tr) >= 1000:
                new_model = make_xgb(DEVICE)
                new_model.fit(X_tr, y_tr)
                model             = new_model
                last_retrain_year = current_year
                print(f"  [{test_date.strftime('%Y-%m')}] Retrained on {len(y_tr):,} obs")
            else:
                print(f"  [{test_date.strftime('%Y-%m')}] WARNING: {len(y_tr):,} obs — keeping prev")

        month_df = kelly_proc[kelly_proc['date'] == test_date].copy()
        if len(month_df) < 20:
            continue
        month_df['pred'] = predict_month(month_df, feature_cols, model)
        monthly_rf       = rf_series.get(test_date, 0.0)

        result = build_portfolio_with_costs(
            month_df, n_deciles, test_date, prev_long, prev_short,
            tc              = transaction_cost,
            rf_rate         = monthly_rf,
            borrow_cost_ann = BORROW_COST_ANNUAL,
            rebate_haircut  = REBATE_HAIRCUT,
            cap_efficiency  = True,
            turnover_buffer = turnover_buffer,
            piotroski_filter= piotroski_filter,
        )
        if result:
            prev_long  = result.pop('long_permnos')
            prev_short = result.pop('short_permnos')
            results.append(result)

    # Extension: train on the full Kelly history, predict 2022-2024
    # This is what a real investor would deploy in January 2022
    print(f"Running extension ({len(ext_dates)} months)...")
    X_full, y_full = prepare_training_data(kelly_proc, feature_cols)
    model_full = make_xgb(DEVICE)
    model_full.fit(X_full, y_full)
    print(f"  Full model trained on {len(y_full):,} obs")

    prev_long  = set()
    prev_short = set()
    for ext_date in ext_dates:
        month_df = ext_proc[ext_proc['date'] == ext_date].copy()
        if len(month_df) < 20:
            continue
        month_df['pred'] = predict_month(month_df, feature_cols, model_full)
        monthly_rf       = rf_series.get(ext_date, 0.0)

        result = build_portfolio_with_costs(
            month_df, n_deciles, ext_date, prev_long, prev_short,
            tc              = transaction_cost,
            rf_rate         = monthly_rf,
            borrow_cost_ann = BORROW_COST_ANNUAL,
            rebate_haircut  = REBATE_HAIRCUT,
            cap_efficiency  = True,
            turnover_buffer = turnover_buffer,
            piotroski_filter= piotroski_filter,
        )
        if result:
            prev_long  = result.pop('long_permnos')
            prev_short = result.pop('short_permnos')
            results.append(result)

    port     = pd.DataFrame(results).set_index('date')
    expected = pd.date_range(port.index.min(), port.index.max(), freq='MS')
    missing  = expected.difference(port.index)
    print(f"\n  Months: {len(port)} | Missing: {len(missing)} | "
          f"Avg TO: {port['avg_turnover'].mean():.1%}")
    return port, model_full

## Steps 10–12 — In-sample backtest, FF5 benchmark, and rolling metric helpers

### In-sample backtest

The in-sample backtest runs the same expanding-window logic over 1962–2014 using the Kelly dataset. It starts in 1962 rather than 1957 to ensure at least 5 years of training data before the first retrain. The results are stored in `insample_allocation.csv` and used only for the overfitting diagnostic — comparing in-sample Sharpe to OOS Sharpe. These returns are **not** suitable for allocation decisions since the model has seen all this data during training.

### FF5 linear benchmark

The FF5 benchmark is the most important comparison. It uses rolling 60-month OLS to estimate each stock's factor loadings, multiplies by current factor realisations to predict returns, then forms the identical decile long-short portfolio with the same TC, borrow, rebate, and capital efficiency. The only difference from our XGBoost models is the linear vs nonlinear prediction.

The GPU implementation uses CuPy for vectorised matrix operations. Rather than running N separate OLS regressions in a loop, the full return matrix (T×N) is sent to GPU once and all stocks are solved simultaneously: B = (X'X)⁻¹X'R where R has one column per stock. This reduces runtime from hours to minutes.

### Rolling metric helpers

Pre-computed rolling metrics (Sharpe, Sortino, Calmar, max drawdown, hit rate) at 12 and 36-month windows are stored in the allocation CSVs. This allows the presentation charts to pull pre-computed values rather than recomputing on every plot.

In [ ]:
# ============================================================
# STEP 10: IN-SAMPLE BACKTEST ENGINE
# Covers 1962-2014 (need 5yr min data before first retrain)
# Uses expanding window retrains — same logic as OOS
# IMPORTANT: these are in-sample, model has seen this data
# Use only for overfitting diagnostics, not allocation
# ============================================================

def run_insample(kelly_df, feature_cols, model_name,
                 is_start         = '1962-01-01',
                 train_end        = TRAIN_END,
                 n_deciles        = 10,
                 transaction_cost = TC_ONEWAY,
                 min_coverage     = 0.0,
                 turnover_buffer  = 0.0):

    print(f"\n{'='*60}")
    print(f"IN-SAMPLE — {model_name} | {len(feature_cols)} features")
    print(f"Period: {is_start} → {train_end}")
    print(f"{'='*60}")

    kelly_f    = apply_coverage_filter(kelly_df, feature_cols, min_coverage)
    kelly_proc = rank_normalize(kelly_f, feature_cols)

    is_dates = sorted(kelly_proc[
        (kelly_proc['date'] >= is_start) &
        (kelly_proc['date'] <= train_end)
    ]['date'].unique())

    results           = []
    model             = None
    last_retrain_year = None
    prev_long         = set()
    prev_short        = set()

    print(f"Running {len(is_dates)} in-sample months...")

    for test_date in is_dates:
        current_year = test_date.year

        if current_year != last_retrain_year:
            train      = kelly_proc[kelly_proc['date'] < test_date]
            X_tr, y_tr = prepare_training_data(train, feature_cols)

            # Skip years with too little training data (early 1960s)
            if len(y_tr) < 500:
                last_retrain_year = current_year
                continue

            new_model = make_xgb(DEVICE)
            new_model.fit(X_tr, y_tr)
            model             = new_model
            last_retrain_year = current_year

            # Print progress every 5 years to avoid log spam
            if current_year % 5 == 0:
                print(f"  [{test_date.strftime('%Y-%m')}] Retrained on {len(y_tr):,} obs")

        if model is None:
            continue

        month_df = kelly_proc[kelly_proc['date'] == test_date].copy()
        if len(month_df) < 20:
            continue

        month_df['pred'] = predict_month(month_df, feature_cols, model)
        monthly_rf       = rf_series.get(test_date, 0.0)

        result = build_portfolio_with_costs(
            month_df, n_deciles, test_date, prev_long, prev_short,
            tc              = transaction_cost,
            rf_rate         = monthly_rf,
            borrow_cost_ann = BORROW_COST_ANNUAL,
            rebate_haircut  = REBATE_HAIRCUT,
            cap_efficiency  = True,
            turnover_buffer = turnover_buffer,
            piotroski_filter= False,
        )
        if result:
            prev_long  = result.pop('long_permnos')
            prev_short = result.pop('short_permnos')
            results.append(result)

    port = pd.DataFrame(results).set_index('date')
    print(f"  Done: {len(port)} months")
    return port

# ============================================================
# STEP 11: FF5 LINEAR BENCHMARK
# ============================================================

def run_ff5_benchmark(kelly_df, ext_df,
                      test_start=TEST_START, test_end=TEST_END,
                      ext_start=EXT_START,   ext_end=EXT_END,
                      rolling_window=60, n_deciles=10,
                      transaction_cost=TC_ONEWAY):

    print("\n" + "="*60)
    print("FF5 Linear Benchmark (Vectorized + CuPy GPU)")
    print(f"Window: {rolling_window}m | TC: {transaction_cost*10000:.0f}bps")
    print("="*60)

    # Try CuPy for GPU-accelerated matrix operations
    # The vectorised approach solves all N stock OLS regressions simultaneously
    # as a single matrix operation rather than N sequential loops
    try:
        import cupy as cp
        from cupy.linalg import inv as cp_inv
        USE_GPU = True
        _ = cp.array([1.0])
        cp.cuda.Stream.null.synchronize()
        print("CuPy — GPU")
    except ImportError:
        USE_GPU = False
        cp = np
        cp_inv = np.linalg.inv
        print("NumPy — CPU")

    # Build the full return matrix (T × N) — uploaded to GPU once at startup
    all_rets   = pd.concat([kelly_df[['permno','date','ret']],
                             ext_df[['permno','date','ret']]], ignore_index=True)
    ret_matrix = all_rets.pivot(index='date', columns='permno', values='ret').sort_index()
    print(f"Return matrix: {ret_matrix.shape}")

    factors  = ff5[['MKT','SMB','HML','RMW','CMA']].reindex(ret_matrix.index)
    ret_vals = ret_matrix.values.astype(np.float32)
    fac_vals = factors.values.astype(np.float32)

    if USE_GPU:
        ret_gpu = cp.array(ret_vals)
        fac_gpu = cp.array(fac_vals)
        ones_w  = cp.ones((rolling_window, 1), dtype=cp.float32)
    else:
        ret_gpu = ret_vals
        fac_gpu = fac_vals
        ones_w  = np.ones((rolling_window, 1), dtype=np.float32)

    permnos    = ret_matrix.columns.values
    date_index = ret_matrix.index
    all_dates  = [d for d in pd.date_range(test_start, ext_end, freq='MS')
                  if d in date_index]

    results    = []
    prev_long  = set()
    prev_short = set()

    print(f"Processing {len(all_dates)} months...")
    for t_idx, test_date in enumerate(all_dates):
        if t_idx % 12 == 0:
            print(f"  [{test_date.strftime('%Y-%m')}]")

        date_pos = date_index.get_loc(test_date)
        if date_pos < rolling_window:
            continue

        # Slice the rolling 60-month window for this test date
        w  = slice(date_pos - rolling_window, date_pos)
        F  = fac_gpu[w]
        R  = ret_gpu[w]
        X  = cp.hstack([ones_w, F]) if USE_GPU else np.hstack([ones_w, F])

        # Solve B = (X'X)^{-1} X'R — all N stocks simultaneously
        try:
            XtX     = X.T @ X
            XtX_inv = cp_inv(XtX)
            B       = XtX_inv @ (X.T @ R)
        except Exception:
            continue

        curr_f = fac_vals[date_pos]
        curr_X = np.array([1.0, *curr_f], dtype=np.float32)

        if USE_GPU:
            pred_np      = cp.asnumpy(cp.array(curr_X) @ B)
            nan_count    = cp.isnan(R).sum(axis=0)
            curr_rets_np = cp.asnumpy(ret_gpu[date_pos])
            valid_np     = cp.asnumpy(
                ((rolling_window - nan_count) >= 24) &
                ~cp.isnan(ret_gpu[date_pos])
            )
        else:
            pred_np      = curr_X @ B
            nan_count    = np.isnan(R).sum(axis=0)
            curr_rets_np = ret_vals[date_pos]
            valid_np     = ((rolling_window - nan_count) >= 24) & ~np.isnan(curr_rets_np)

        valid_idx = np.where(valid_np)[0]
        if len(valid_idx) < 50:
            continue

        pred_df = pd.DataFrame({
            'permno': permnos[valid_idx],
            'pred':   pred_np[valid_idx],
            'ret':    curr_rets_np[valid_idx],
        })

        try:
            pred_df['decile'] = pd.qcut(pred_df['pred'], n_deciles,
                                         labels=False, duplicates='drop')
        except ValueError:
            continue

        actual_max = pred_df['decile'].max()
        actual_min = pred_df['decile'].min()
        if actual_max == actual_min:
            continue

        long_df  = pred_df[pred_df['decile'] == actual_max]
        short_df = pred_df[pred_df['decile'] == actual_min]
        if len(long_df) == 0:
            continue

        long_gross  = long_df['ret'].mean()
        short_gross = short_df['ret'].mean() if len(short_df) > 0 else 0.0

        curr_long  = set(long_df['permno'].values)
        curr_short = set(short_df['permno'].values)
        long_to    = len(curr_long  - prev_long)  / len(curr_long)  if prev_long  else 1.0
        short_to   = len(curr_short - prev_short) / len(curr_short) if prev_short else 1.0

        monthly_rf  = rf_series.get(test_date, 0.0)
        net_rebate  = max(monthly_rf - REBATE_HAIRCUT/12, 0.0)
        short_pnl   = (-short_gross + net_rebate
                       - BORROW_COST_ANNUAL/12 - short_to * transaction_cost)
        long_net    = long_gross - long_to * transaction_cost
        ls_net_pre  = long_net + short_pnl
        ls_net      = ls_net_pre * CAPITAL_EFFICIENCY

        results.append({
            'date':                  test_date,
            'long_ret':              long_net,
            'short_pnl':             short_pnl,
            'ls_ret':                ls_net,
            'ls_ret_gross':          long_gross - short_gross,
            'ls_ret_pre_capeff':     ls_net_pre,
            'long_turnover':         long_to,
            'short_turnover':        short_to,
            'avg_turnover':          (long_to + short_to) / 2,
            'n_long':                len(long_df),
            'n_short':               len(short_df),
            'mkt_ret':               pred_df['ret'].mean(),
            'avg_fscore_long':       np.nan,
            'avg_fscore_short':      np.nan,
            'tc_cost_monthly':       (long_to + short_to) * transaction_cost,
            'borrow_cost_monthly':   BORROW_COST_ANNUAL / 12,
            'rebate_monthly':        net_rebate,
            'net_financing_monthly': net_rebate - BORROW_COST_ANNUAL/12,
            'rf_rate':               monthly_rf,
        })
        prev_long  = curr_long
        prev_short = curr_short

    if USE_GPU:
        del ret_gpu, fac_gpu, ones_w
        cp.get_default_memory_pool().free_all_blocks()

    port = pd.DataFrame(results).set_index('date')
    print(f"Done: {len(port)} months | avg TO: {port['avg_turnover'].mean():.1%}")
    return port

# ============================================================
# STEP 12: ROLLING METRIC HELPERS
# Pre-computed and stored in the allocation CSVs
# so presentation charts do not need to recompute them
# ============================================================

def rolling_cumret(s, w):
    return s.rolling(w, min_periods=w//2).apply(
        lambda x: (1+x).prod()-1, raw=False)

def rolling_sharpe(s, rf, w):
    ex = s - rf.reindex(s.index).fillna(0)
    return (ex.rolling(w, min_periods=w//2).mean() /
            ex.rolling(w, min_periods=w//2).std() * np.sqrt(12))

def rolling_sortino(s, rf, w):
    ex  = s - rf.reindex(s.index).fillna(0)
    dn  = ex.clip(upper=0).rolling(w, min_periods=w//2).std()
    return (ex.rolling(w, min_periods=w//2).mean() / dn * np.sqrt(12))

def rolling_maxdd(s, w):
    def _mdd(x):
        c = (1+x).cumprod()
        return ((c - c.cummax()) / c.cummax()).min()
    return s.rolling(w, min_periods=w//2).apply(_mdd, raw=False)

def rolling_calmar(s, w):
    ann = s.rolling(w, min_periods=w//2).apply(
        lambda x: (1+x).prod()**(12/len(x))-1, raw=False)
    mdd = rolling_maxdd(s, w).abs()
    return ann / mdd.replace(0, np.nan)

def rolling_hitrate(s, w):
    return (s > 0).rolling(w, min_periods=w//2).mean()

## Step 13 — Master CSV builder

This function takes the portfolio results dictionaries and builds the allocation CSVs that every downstream analysis reads from. It pre-computes all rolling metrics (Sharpe, Sortino, Calmar, drawdown, hit rates at 12 and 36-month windows) and stores them alongside the raw monthly returns.

For the OOS CSV it also adds an HML Passive row — the simple passive value factor treated as a benchmark with no transaction costs. HML is included in the OOS output only, not the in-sample output.

The `period` column (`oos` or `in_sample`) allows both CSVs to be stacked into `full_history_allocation.csv` while remaining filterable.

In [ ]:
# ============================================================
# STEP 13: MASTER CSV BUILDER
# ============================================================

def build_allocation_csv(port_dict, rf_series, hml, period='oos'):
    """
    Builds the master allocation CSV with all rolling metrics pre-computed.
    period: 'oos' or 'in_sample' — written into the 'period' column.
    HML Passive is only added for OOS — it is a benchmark, not a strategy.
    """
    all_rows = []
    strats   = dict(port_dict)

    # Add HML Passive for OOS only — a pure passive value factor with no costs
    if period == 'oos':
        hml_full = hml.reindex(
            pd.date_range('2015-01-01', '2024-11-01', freq='MS')
        ).dropna()
        hml_df = pd.DataFrame({
            'ls_ret':                hml_full,
            'ls_ret_gross':          hml_full,
            'ls_ret_pre_capeff':     hml_full,
            'long_ret':              hml_full,
            'short_pnl':             pd.Series(0.0, index=hml_full.index),
            'long_turnover':         np.nan,
            'short_turnover':        np.nan,
            'avg_turnover':          np.nan,
            'n_long':                np.nan,
            'n_short':               np.nan,
            'mkt_ret':               np.nan,
            'avg_fscore_long':       np.nan,
            'avg_fscore_short':      np.nan,
            'tc_cost_monthly':       np.nan,
            'borrow_cost_monthly':   np.nan,
            'rebate_monthly':        np.nan,
            'net_financing_monthly': np.nan,
            'rf_rate':               rf_series.reindex(hml_full.index).fillna(0),
        })
        strats['HML_Passive'] = hml_df

    for strat_name, port in strats.items():
        print(f"  Rolling metrics: {strat_name} ({period})...")
        rets = port['ls_ret'].copy()

        nav         = (1 + rets).cumprod()
        ret_1m      = rets
        ret_3m      = rolling_cumret(rets, 3)
        ret_6m      = rolling_cumret(rets, 6)
        ret_12m     = rolling_cumret(rets, 12)
        ret_24m     = rolling_cumret(rets, 24)
        ret_36m     = rolling_cumret(rets, 36)
        vol_12m     = rets.rolling(12, min_periods=6).std()  * np.sqrt(12)
        vol_36m     = rets.rolling(36, min_periods=18).std() * np.sqrt(12)
        mdd_6m      = rolling_maxdd(rets, 6)
        mdd_12m     = rolling_maxdd(rets, 12)
        mdd_36m     = rolling_maxdd(rets, 36)
        sharpe_12m  = rolling_sharpe(rets, rf_series, 12)
        sharpe_36m  = rolling_sharpe(rets, rf_series, 36)
        sortino_12m = rolling_sortino(rets, rf_series, 12)
        sortino_36m = rolling_sortino(rets, rf_series, 36)
        calmar_12m  = rolling_calmar(rets, 12)
        calmar_36m  = rolling_calmar(rets, 36)
        hit_3m      = rolling_hitrate(rets, 3)
        hit_6m      = rolling_hitrate(rets, 6)
        hit_12m     = rolling_hitrate(rets, 12)

        def g(series, date):
            try:   return series.loc[date]
            except: return np.nan

        def gc(col, date):
            if col not in port.columns:
                return np.nan
            try:   return port.loc[date, col]
            except: return np.nan

        for date in rets.index:
            all_rows.append({
                'date':                  date,
                'strategy':              strat_name,
                'period':                period,
                'ls_ret':                g(rets, date),
                'ls_ret_gross':          gc('ls_ret_gross', date),
                'ls_ret_pre_capeff':     gc('ls_ret_pre_capeff', date),
                'long_ret':              gc('long_ret', date),
                'short_pnl':             gc('short_pnl', date),
                'long_turnover':         gc('long_turnover', date),
                'short_turnover':        gc('short_turnover', date),
                'avg_turnover':          gc('avg_turnover', date),
                'n_long':                gc('n_long', date),
                'n_short':               gc('n_short', date),
                'mkt_ret':               gc('mkt_ret', date),
                'avg_fscore_long':       gc('avg_fscore_long', date),
                'avg_fscore_short':      gc('avg_fscore_short', date),
                'tc_cost_monthly':       gc('tc_cost_monthly', date),
                'borrow_cost_monthly':   gc('borrow_cost_monthly', date),
                'rebate_monthly':        gc('rebate_monthly', date),
                'net_financing_monthly': gc('net_financing_monthly', date),
                'rf_rate':               gc('rf_rate', date),
                'nav':                   g(nav, date),
                'ret_1m':   g(ret_1m,  date), 'ret_3m':  g(ret_3m,  date),
                'ret_6m':   g(ret_6m,  date), 'ret_12m': g(ret_12m, date),
                'ret_24m':  g(ret_24m, date), 'ret_36m': g(ret_36m, date),
                'vol_12m':  g(vol_12m, date), 'vol_36m': g(vol_36m, date),
                'mdd_6m':   g(mdd_6m,  date), 'mdd_12m': g(mdd_12m, date), 'mdd_36m': g(mdd_36m, date),
                'sharpe_12m':  g(sharpe_12m,  date), 'sharpe_36m':  g(sharpe_36m,  date),
                'sortino_12m': g(sortino_12m, date), 'sortino_36m': g(sortino_36m, date),
                'calmar_12m':  g(calmar_12m,  date), 'calmar_36m':  g(calmar_36m,  date),
                'hit_rate_3m': g(hit_3m, date), 'hit_rate_6m': g(hit_6m, date),
                'hit_rate_12m': g(hit_12m, date),
            })

    return pd.DataFrame(all_rows)

## Steps 14 — Performance reporting helpers

Two reporting functions used for the summary tables and the regime analysis. `compute_metrics` produces the headline statistics for the full period. `compute_yearly_metrics` produces year-by-year breakdowns used in the heatmap and bar charts. The yearly breakdown is the most important honesty check — it shows flat or negative years like 2017–2019 alongside the strong years like 2021, preventing the jury from thinking we cherry-picked the reporting period.

In [ ]:
# ============================================================
# STEP 14: PERFORMANCE REPORTING HELPERS
# ============================================================

def compute_metrics(returns, name='Strategy', rf=None):
    returns   = returns.dropna()
    rf_a      = rf.reindex(returns.index).fillna(0) if rf is not None \
                else pd.Series(0, index=returns.index)
    excess    = returns - rf_a
    ann_ret   = (1+returns).prod() ** (12/len(returns)) - 1
    ann_vol   = returns.std() * np.sqrt(12)
    sharpe    = excess.mean() / excess.std() * np.sqrt(12)
    cum       = (1+returns).cumprod()
    max_dd    = ((cum - cum.cummax()) / cum.cummax()).min()
    calmar    = ann_ret / abs(max_dd) if max_dd != 0 else np.nan
    hit_rate  = (returns > 0).mean()
    return {
        'Name':       name,
        'Ann Return': f"{ann_ret:.2%}",
        'Ann Vol':    f"{ann_vol:.2%}",
        'Sharpe':     f"{sharpe:.2f}",
        'Max DD':     f"{max_dd:.2%}",
        'Calmar':     f"{calmar:.2f}",
        'Hit Rate':   f"{hit_rate:.1%}",
    }

def compute_yearly_metrics(returns, name='Strategy', rf=None):
    # Year-by-year breakdown — the most important honesty check in the presentation
    # Shows flat/negative years alongside strong years so no one thinks we cherry-picked
    returns  = returns.dropna()
    rf_a     = rf.reindex(returns.index).fillna(0) if rf is not None \
               else pd.Series(0, index=returns.index)
    rows = []
    for year in sorted(returns.index.year.unique()):
        yr  = returns[returns.index.year == year]
        yrf = rf_a[rf_a.index.year == year]
        if len(yr) < 3: continue
        ex  = yr - yrf
        rows.append({
            'Year':       year,
            'Name':       name,
            'Ann Return': (1+yr).prod() - 1,
            'Sharpe':     ex.mean()/ex.std()*np.sqrt(12) if ex.std()>0 else np.nan,
            'Max DD':     (((1+yr).cumprod() / (1+yr).cumprod().cummax()) - 1).min(),
        })
    return pd.DataFrame(rows)

## Steps 15–16 — Run all backtests

This is where the pipeline actually executes. Steps 15 runs all four ML models plus the FF5 benchmark through the OOS backtest. Step 16 runs the in-sample backtest for all four models.

Each model run is independent — the loop iterates over `FEATURE_SETS` and calls `run_backtest` with the per-model settings from `COVERAGE` and `TURNOVER_BUFFER`. The final model (trained on all Kelly data) is stored in `final_models` for the feature importance analysis in Step 22.

Total runtime on GPU: approximately 2–3 hours for all four models plus FF5. On CPU: 8–12 hours.

In [ ]:
# ============================================================
# STEP 15: RUN OOS BACKTESTS
# ============================================================

portfolios   = {}
final_models = {}

for model_name, feature_cols in FEATURE_SETS.items():
    port, model = run_backtest(
        kelly, ext, feature_cols, model_name,
        min_coverage     = COVERAGE[model_name],
        turnover_buffer  = TURNOVER_BUFFER[model_name],
        piotroski_filter = False,
    )
    portfolios[model_name]   = port
    final_models[model_name] = model
    gc.collect()

# FF5 linear benchmark — same cost model as our strategies
# The only difference is linear OLS predictions vs our nonlinear trees
ff5_port = run_ff5_benchmark(kelly, ext)
portfolios['FF5_Linear'] = ff5_port

print("\nAll OOS models complete.")

In [ ]:
# ============================================================
# STEP 16: RUN IN-SAMPLE BACKTESTS
# ============================================================
# Run all four ML models in-sample for the overfitting diagnostic
# FF5 is excluded — it has no in-sample equivalent in this framework

IS_MODELS     = ['A_Value', 'B_ValueQuality', 'C_ValueQualityMom', 'D_ValMOM']
is_portfolios = {}

for model_name in IS_MODELS:
    feature_cols = FEATURE_SETS[model_name]
    port_is = run_insample(
        kelly, feature_cols, model_name,
        train_end       = TRAIN_END,
        min_coverage    = COVERAGE[model_name],
        turnover_buffer = TURNOVER_BUFFER[model_name],
    )
    is_portfolios[model_name] = port_is
    gc.collect()

print("\nAll in-sample models complete.")

## Step 17 — Build and save output CSVs

Three files are produced:
- `master_allocation.csv` — OOS results for all 5 strategies plus HML Passive, 119 months × 6 strategies = 714 rows
- `insample_allocation.csv` — in-sample results for the 4 ML models, 1962–2014
- `full_history_allocation.csv` — both periods stacked, filterable by the `period` column

All downstream analysis and presentation charts read from these three files. The pipeline only needs to be rerun once — after that, any chart or table can be reproduced from the saved CSVs without rerunning the 3-hour backtest.

In [ ]:
# ============================================================
# STEP 17: BUILD AND SAVE CSVs
# ============================================================

hml = ff5['HML']

print("\nBuilding OOS master CSV...")
master_oos = build_allocation_csv(portfolios, rf_series, hml, period='oos')
master_oos.to_csv('master_allocation.csv', index=False)
print(f"Saved: master_allocation.csv  "
      f"({master_oos.shape[0]:,} rows × {master_oos.shape[1]} cols)")

print("\nBuilding in-sample CSV...")
master_is = build_allocation_csv(is_portfolios, rf_series, hml, period='in_sample')
master_is.to_csv('insample_allocation.csv', index=False)
print(f"Saved: insample_allocation.csv  "
      f"({master_is.shape[0]:,} rows × {master_is.shape[1]} cols)")

print("\nBuilding combined full history CSV...")
common_cols  = [c for c in master_oos.columns if c in master_is.columns]
full_history = pd.concat(
    [master_oos[common_cols], master_is[common_cols]],
    ignore_index=True
)
full_history = full_history.sort_values(['strategy','date']).reset_index(drop=True)
full_history.to_csv('full_history_allocation.csv', index=False)
print(f"Saved: full_history_allocation.csv  "
      f"({full_history.shape[0]:,} rows × {full_history.shape[1]} cols)")
print(f"  Date range: {full_history['date'].min()} → {full_history['date'].max()}")
print(f"  Strategies: {sorted(full_history['strategy'].unique())}")

## Steps 18–23 — Performance reporting, analysis, and plots

The remaining cells produce the printed summaries and charts used in the presentation. They all read from the already-computed `portfolios` dict and do not require rerunning the backtest.

In [ ]:
# ============================================================
# STEP 18: PERFORMANCE SUMMARY
# ============================================================

print("\n" + "="*70)
print("PERFORMANCE SUMMARY — OOS (net, full cost model)")
print("="*70)
rows = []
for name, port in portfolios.items():
    rows.append(compute_metrics(port['ls_ret'], name=f"{name}_net", rf=rf_series))
    if 'ls_ret_gross' in port.columns:
        rows.append(compute_metrics(port['ls_ret_gross'], name=f"{name}_gross", rf=rf_series))
hml_oos = hml.reindex(portfolios['A_Value'].index).dropna()
rows.append(compute_metrics(hml_oos, name='HML_Passive', rf=rf_series))
print(pd.DataFrame(rows).set_index('Name').to_string())

print("\nTurnover & Cost breakdown:")
print(f"{'Model':<25} {'Avg TO':>8} {'TC/yr':>8} {'Borrow/yr':>10} "
      f"{'Rebate/yr':>10} {'Net fin/yr':>12}")
print("-"*75)
for name, port in portfolios.items():
    if 'avg_turnover' not in port.columns: continue
    avg_to = port['avg_turnover'].mean()
    tc_yr  = port['tc_cost_monthly'].mean()     * 12 * 100 if 'tc_cost_monthly'     in port.columns else 0
    bor_yr = port['borrow_cost_monthly'].mean() * 12 * 100 if 'borrow_cost_monthly' in port.columns else 0
    reb_yr = port['rebate_monthly'].mean()      * 12 * 100 if 'rebate_monthly'      in port.columns else 0
    fin_yr = port['net_financing_monthly'].mean()*12 * 100 if 'net_financing_monthly' in port.columns else 0
    print(f"{name:<25} {avg_to:>7.1%} {tc_yr:>7.2f}% {bor_yr:>9.2f}% "
          f"{reb_yr:>9.2f}% {fin_yr:>11.2f}%")

In [ ]:
# ============================================================
# STEP 19: REGIME & YEARLY ANALYSIS
# ============================================================

regimes = {
    'Full OOS (2015-2024)':    (TEST_START, '2024-11-01'),
    'Lost Decade (2015-2020)': ('2015-01-01','2020-12-01'),
    'Post-COVID (2020-2022)':  ('2020-01-01','2022-12-01'),
    'Extension (2022-2024)':   (EXT_START,   EXT_END),
}

print("\n" + "="*70)
print("REGIME ANALYSIS")
print("="*70)
regime_rows = []
for regime, (s, e) in regimes.items():
    for name, port in portfolios.items():
        rets = port['ls_ret'][s:e].dropna()
        if len(rets) < 3: continue
        m = compute_metrics(rets, name=name, rf=rf_series)
        m['Regime'] = regime
        regime_rows.append(m)
    hml_r = hml[s:e].dropna()
    if len(hml_r) >= 3:
        m = compute_metrics(hml_r, name='HML_Passive', rf=rf_series)
        m['Regime'] = regime
        regime_rows.append(m)
regime_df = (pd.DataFrame(regime_rows)
             .set_index(['Regime','Name'])
             [['Ann Return','Sharpe','Max DD']])
print(regime_df.to_string())

print("\n" + "="*70)
print("YEARLY METRICS")
print("="*70)
all_yr = []
for name, port in portfolios.items():
    all_yr.append(compute_yearly_metrics(port['ls_ret'], name=name, rf=rf_series))
hml_full = hml.reindex(pd.date_range('2015-01-01','2024-11-01',freq='MS')).dropna()
all_yr.append(compute_yearly_metrics(hml_full, name='HML_Passive', rf=rf_series))
yearly_df     = pd.concat(all_yr, ignore_index=True)
yearly_return = yearly_df.pivot(index='Year', columns='Name', values='Ann Return')
yearly_sharpe = yearly_df.pivot(index='Year', columns='Name', values='Sharpe')
print("\nYearly Returns:")
print(yearly_return.applymap(lambda x: f"{x:.1%}" if pd.notna(x) else "—").to_string())
print("\nYearly Sharpe:")
print(yearly_sharpe.applymap(lambda x: f"{x:.2f}" if pd.notna(x) else "—").to_string())
yearly_return.to_csv('yearly_returns.csv')
yearly_sharpe.to_csv('yearly_sharpe.csv')

In [ ]:
# ============================================================
# STEP 20: OVERFITTING CHECK
# ============================================================
# Compare in-sample Sharpe (1962-2014) vs OOS Sharpe (2015-2024)
# A ratio IS/OOS < 1 means OOS beats IS — the opposite of overfitting
# A ratio > 3 would be a red flag suggesting the model fit the training noise

print("\n" + "="*60)
print("OVERFITTING CHECK: In-sample vs OOS")
print("="*60)
print(f"{'Model':<25} {'IS Sharpe':>10} {'OOS Sharpe':>11} {'Ratio':>8}")
print("-"*56)
for name in IS_MODELS:
    if name not in is_portfolios or name not in portfolios: continue
    is_r   = is_portfolios[name]['ls_ret']
    oos_r  = portfolios[name]['ls_ret']
    is_rf  = rf_series.reindex(is_r.index).fillna(0)
    oos_rf = rf_series.reindex(oos_r.index).fillna(0)
    is_sr  = ((is_r  - is_rf).mean()  / (is_r  - is_rf).std())  * np.sqrt(12)
    oos_sr = ((oos_r - oos_rf).mean() / (oos_r - oos_rf).std()) * np.sqrt(12)
    ratio  = is_sr / oos_sr if oos_sr != 0 else np.nan
    flag   = "✓ OK" if 1.0 < ratio < 3.0 else "⚠ CHECK"
    print(f"{name:<25} {is_sr:>10.2f} {oos_sr:>11.2f} {ratio:>8.2f}  {flag}")

In [ ]:
# ============================================================
# STEP 21: ALPHA vs FF5
# ============================================================
# Regress each strategy's monthly returns on the 5 FF5 factors
# with HAC standard errors (6 lags) to correct for serial correlation
# Alpha = the return unexplained by any of the 5 known risk factors
# t > 3 = statistically significant at the 1% level with 119 months of data

print("\n" + "="*70)
print("ALPHA vs FF5 (HAC, 6 lags)")
print("="*70)
for name, port in portfolios.items():
    y   = port['ls_ret'].reindex(ff5.index).dropna()
    X   = ff5[['MKT','SMB','HML','RMW','CMA']].reindex(y.index).dropna()
    y   = y.reindex(X.index)
    res = sm.OLS(y, sm.add_constant(X)).fit(
              cov_type='HAC', cov_kwds={'maxlags':6})
    t  = res.tvalues['const']
    st = '***' if abs(t)>3 else '**' if abs(t)>2 else '*' if abs(t)>1.65 else ''
    print(f"\n{name}:")
    print(f"  Alpha:  {res.params['const']*12:.2%}/yr  (t={t:.2f}{st})")
    print(f"  HML:    {res.params['HML']:.3f}  MKT: {res.params['MKT']:.3f}  R²: {res.rsquared:.3f}")

In [ ]:
# ============================================================
# STEP 22: FEATURE IMPORTANCE
# ============================================================
# XGBoost feature importance = total gain contributed by each feature
# across all splits in all trees in the final model (trained on full Kelly dataset)
# dy dominating Model A at ~65% is a key result — Model A is essentially
# a high-dividend-yield strategy. Models B and D are better diversified.

print("\n" + "="*70)
print("FEATURE IMPORTANCE")
print("="*70)
for model_key, feature_cols in FEATURE_SETS.items():
    if model_key not in final_models: continue
    imp = pd.Series(
        final_models[model_key].feature_importances_,
        index=feature_cols
    ).sort_values(ascending=False)
    print(f"\n{model_key} — Top 10:")
    print(imp.head(10).to_string())

In [ ]:
# ============================================================
# STEP 23: PLOTS
# ============================================================

colors_strat = {
    'A_Value':           '#B0BEC5',
    'B_ValueQuality':    '#4CAF50',
    'C_ValueQualityMom': '#2196F3',
    'D_ValMOM':          '#FF9800',
    'FF5_Linear':        '#9C27B0',
}

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle(f'ML Value Strategy — OOS Results ({TEST_START[:4]}–2024)',
             fontsize=14, fontweight='bold')

# Panel 1: equity curves — the core result visual
ax = axes[0,0]
for name, port in portfolios.items():
    (1 + port['ls_ret']).cumprod().plot(
        ax=ax, label=f"{name} (net)",
        color=colors_strat.get(name,'#999'), lw=1.8)
(1 + hml.reindex(portfolios['A_Value'].index).dropna()).cumprod().plot(
    ax=ax, label='HML Passive', color='#E53935', lw=1.8, ls='--')
ax.axvline(pd.Timestamp(EXT_START), color='black', lw=1, ls=':', alpha=0.6)
ax.set_title('Equity Curves ($1, net of all costs)', fontweight='bold')
ax.legend(fontsize=7); ax.grid(alpha=0.2)

# Panel 2: drawdown
ax = axes[0,1]
for name, port in portfolios.items():
    cum = (1 + port['ls_ret']).cumprod()
    ((cum - cum.cummax()) / cum.cummax()).plot(
        ax=ax, label=name, color=colors_strat.get(name,'#999'), lw=1.5)
hml_s = hml.reindex(portfolios['A_Value'].index).dropna()
cum_h = (1 + hml_s).cumprod()
((cum_h - cum_h.cummax()) / cum_h.cummax()).plot(
    ax=ax, label='HML Passive', color='#E53935', lw=1.5, ls='--')
ax.set_title('Drawdown', fontweight='bold')
ax.legend(fontsize=7); ax.grid(alpha=0.2)

# Panel 3: yearly returns heatmap
ax = axes[1,0]
plot_cols = [c for c in ['A_Value','B_ValueQuality','C_ValueQualityMom',
                          'D_ValMOM','FF5_Linear','HML_Passive']
             if c in yearly_return.columns]
yr_plot = yearly_return[plot_cols].dropna(how='all')
im = ax.imshow(yr_plot.T.values, aspect='auto', cmap='RdYlGn', vmin=-0.3, vmax=0.3)
ax.set_xticks(range(len(yr_plot.index)))
ax.set_xticklabels(yr_plot.index, rotation=45, fontsize=7)
ax.set_yticks(range(len(plot_cols))); ax.set_yticklabels(plot_cols, fontsize=7)
for i in range(len(plot_cols)):
    for j in range(len(yr_plot.index)):
        val = yr_plot.iloc[j,i]
        if pd.notna(val):
            ax.text(j, i, f"{val:.0%}", ha='center', va='center',
                    fontsize=6, color='black')
ax.set_title('Yearly Returns Heatmap', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)

# Panel 4: yearly Sharpe bars
ax = axes[1,1]
x = np.arange(len(yearly_sharpe.index))
w = 0.13
for i, col in enumerate(plot_cols):
    if col in yearly_sharpe.columns:
        vals = yearly_sharpe[col].reindex(yearly_sharpe.index).fillna(0)
        ax.bar(x + i*w, vals, w, label=col,
               color=colors_strat.get(col,'#E53935'), alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x + w*2.5)
ax.set_xticklabels(yearly_sharpe.index, rotation=45, fontsize=7)
ax.set_title('Yearly Sharpe Ratios', fontweight='bold')
ax.legend(fontsize=6); ax.grid(alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig('ml_value_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# SUMMARY: OUTPUT FILES AND COST MODEL APPLIED
# ============================================================

print("\n" + "="*70)
print("ALL DONE")
print("="*70)
print("Output files:")
for f in ['master_allocation.csv',
          'insample_allocation.csv',
          'full_history_allocation.csv',
          'yearly_returns.csv',
          'yearly_sharpe.csv',
          'ml_value_results.png']:
    print(f"  {f}")
print("\nCost model applied:")
print(f"  TC one-way:       {TC_ONEWAY*10000:.0f} bps")
print(f"  Stock borrow:     {BORROW_COST_ANNUAL*100:.0f} bps/yr")
print(f"  Rebate haircut:   {REBATE_HAIRCUT*100:.0f} bps/yr")
print(f"  Capital factor:   {CAPITAL_EFFICIENCY:.3f}  (Reg T 50% margin → $1.50 capital per $1 L/S)")

---
# PART 2 — POST-RUN ANALYSIS

The cells below are standalone analysis scripts added after the main pipeline was complete. They read from the saved output CSVs and can be run independently without re-running the backtest.

- **Cell A** — Heatmap performance table with IVE and VTV ETF alpha columns
- **Cell B** — Equity curves and drawdown with IVE/VTV as additional benchmarks
- **Cell C** — Short book market cap diagnostic (motivates the AUM capacity analysis)
- **Cell D** — AUM capacity analysis: the 4-step calculation producing the Slide 17 numbers
- **Cell E** — Full AUM capacity chart with sensitivity table and revenue model

---

## Cell A — Performance heatmap with IVE and VTV alpha

Extends the main performance table to include alpha computed against two long-only value ETF benchmarks: IVE (iShares S&P 500 Value) and VTV (Vanguard Value ETF). These provide a practitioner-relevant benchmark alongside the academic FF5 regression alpha.

The Jensen alpha against IVE is computed as: regress `(strategy − RF)` on `(IVE − RF)`, using HAC standard errors. The resulting alpha answers the question a real investor would ask: "Can you do better than just buying a value ETF?"

The heatmap colouring normalises each column independently — best value in each column is darkest green, worst is darkest red. For HML loading and MKT beta, closer to zero is greener since we want market-neutral strategies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import statsmodels.api as sm
import requests, zipfile, io
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

# Load the saved allocation CSV — no need to rerun the backtest
master = pd.read_csv('master_allocation.csv', parse_dates=['date'])

def get_rets(strat):
    return (master[master['strategy']==strat]
            .set_index('date')['ls_ret']
            .sort_index())

rets = {s: get_rets(s) for s in master['strategy'].unique()}

# Pull FF5 factors for alpha regressions and RF series
url = ("https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/"
       "ftp/F-F_Research_Data_5_Factors_2x3_CSV.zip")
r   = requests.get(url)
z   = zipfile.ZipFile(io.BytesIO(r.content))
with z.open(z.namelist()[0]) as f:
    raw = f.read().decode('utf-8')
lines = raw.split('\n')
start = 0
for i, line in enumerate(lines):
    s = line.strip()
    if s and s[0].isdigit() and len(s.split(',')[0].strip()) == 6:
        start = i
        break
ff5 = pd.read_csv(
    StringIO('\n'.join(lines[start:])),
    header=None,
    names=['date','MKT','SMB','HML','RMW','CMA','RF'],
    on_bad_lines='skip'
)
ff5 = ff5[ff5['date'].astype(str).str.strip().str.len()==6]
ff5['date'] = pd.to_datetime(
    ff5['date'].astype(str).str.strip(), format='%Y%m')
ff5 = ff5.set_index('date').apply(pd.to_numeric, errors='coerce') / 100
ff5 = ff5.dropna()

rf  = ff5['RF']
hml = ff5['HML']

idx    = rets['A_Value'].index
rf_oos = rf.reindex(idx).fillna(0)

# Metric helpers — each used in the heatmap column calculations

def ann_ret(r):
    r = r.dropna()
    return (1+r).prod() ** (12/len(r)) - 1

def ann_vol(r):
    return r.dropna().std() * np.sqrt(12)

def sharpe(r, rf_s):
    r = r.dropna()
    ex = r - rf_s.reindex(r.index).fillna(0)
    return ex.mean() / ex.std() * np.sqrt(12)

def maxdd(r):
    r = r.dropna()
    cum = (1+r).cumprod()
    return ((cum - cum.cummax()) / cum.cummax()).min()

def info_ratio_vs_ff5(r):
    # IR = active return / tracking error vs FF5 factor-explained return
    # Active return = realised return minus what the factors explain (no alpha)
    r = r.dropna()
    X = ff5[['MKT','SMB','HML','RMW','CMA']].reindex(r.index).dropna()
    r = r.reindex(X.index)
    res    = sm.OLS(r, sm.add_constant(X)).fit()
    fitted = X @ res.params[1:]
    active = r - fitted
    te     = active.std() * np.sqrt(12)
    ar     = active.mean() * 12
    return ar / te if te > 0 else np.nan

def mkt_beta(r):
    r = r.dropna()
    X = ff5[['MKT']].reindex(r.index).dropna()
    r = r.reindex(X.index)
    res = sm.OLS(r, sm.add_constant(X)).fit()
    return res.params['MKT']

def ff5_alpha(r):
    r = r.dropna()
    X = ff5[['MKT','SMB','HML','RMW','CMA']].reindex(r.index).dropna()
    r = r.reindex(X.index)
    res = sm.OLS(r, sm.add_constant(X)).fit(
        cov_type='HAC', cov_kwds={'maxlags':6})
    return res.params['const']*12, res.tvalues['const']

# HML loadings from the main FF5 regression — hardcoded from Step 21 output
hml_loadings = {
    'A_Value':           0.079,
    'B_ValueQuality':    0.174,
    'C_ValueQualityMom': 0.244,
    'D_ValMOM':          0.136,
    'FF5_Linear':       -0.020,
    'HML_Passive':       np.nan,
}

colors_map = {
    'A_Value':           '#9E9E9E',
    'B_ValueQuality':    '#43A047',
    'C_ValueQualityMom': '#1E88E5',
    'D_ValMOM':          '#FF8F00',
    'FF5_Linear':        '#8E24AA',
    'HML_Passive':       '#E53935',
}

strategies = ['A_Value','B_ValueQuality','C_ValueQualityMom',
              'D_ValMOM','FF5_Linear','HML_Passive']
labels     = ['A — Value Only','B — Value+Quality',
              'C — Val+Quality+Mom','D — Val+Momentum',
              'FF5 Linear','HML Passive']

metrics_data = {}
for strat in strategies:
    r = rets.get(strat, pd.Series(dtype=float))
    if len(r) == 0:
        metrics_data[strat] = {}
        continue
    alpha, tstat = ff5_alpha(r) if strat != 'HML_Passive' \
                   else (np.nan, np.nan)
    stars = ('***' if not np.isnan(tstat) and abs(tstat)>3 else
             '**'  if not np.isnan(tstat) and abs(tstat)>2 else
             '*'   if not np.isnan(tstat) and abs(tstat)>1.65 else '')
    metrics_data[strat] = {
        'ann_ret':  ann_ret(r),
        'ann_vol':  ann_vol(r),
        'sharpe':   sharpe(r, rf),
        'max_dd':   maxdd(r),
        'ir_ff5':   info_ratio_vs_ff5(r) if strat != 'HML_Passive' else np.nan,
        'alpha':    alpha,
        'tstat':    tstat,
        'stars':    stars,
        'hml_load': hml_loadings.get(strat, np.nan),
        'mkt_beta': mkt_beta(r),
    }

# Column definitions: (key, display_name, format_string, higher_is_better)
columns = [
    ('ann_ret',  'Ann\nReturn',   '{:.1%}',  True),
    ('ann_vol',  'Ann\nVol',      '{:.1%}',  False),
    ('sharpe',   'Sharpe',        '{:.2f}',  True),
    ('max_dd',   'Max\nDD',       '{:.1%}',  True),
    ('ir_ff5',   'IR vs\nFF5',    '{:.2f}',  True),
    ('alpha',    'Alpha\nvs FF5', '{:.1%}',  True),
    ('tstat',    't-stat',        '{:.2f}',  True),
    ('hml_load', 'HML\nLoad',     '{:.3f}',  False),
    ('mkt_beta', 'MKT\nBeta',     '{:.3f}',  False),
]

n_strats = len(strategies)
n_cols   = len(columns)
val_mat  = np.full((n_strats, n_cols), np.nan)
txt_mat  = [[''] * n_cols for _ in range(n_strats)]

for i, strat in enumerate(strategies):
    md = metrics_data[strat]
    if not md: continue
    for j, (key, _, fmt, _) in enumerate(columns):
        v = md.get(key, np.nan)
        val_mat[i, j] = v if v is not None else np.nan
        if isinstance(v, float) and np.isnan(v):
            txt_mat[i][j] = '—'
        else:
            txt = fmt.format(v)
            if key == 'tstat':
                txt = txt + md.get('stars','')
            txt_mat[i][j] = txt

# Normalise each column independently for colouring
# HML load and MKT beta: closer to zero = greener (market-neutral is better)
color_mat = np.full((n_strats, n_cols), 0.5)
for j, (key, _, _, hib) in enumerate(columns):
    col_vals = val_mat[:, j].copy()
    valid    = ~np.isnan(col_vals)
    if valid.sum() < 2: continue
    if key in ('hml_load', 'mkt_beta'):
        abs_vals = np.abs(col_vals)
        abs_vals[~valid] = np.nanmax(abs_vals[valid])
        mn, mx = np.nanmin(abs_vals), np.nanmax(abs_vals)
        if mx > mn:
            color_mat[:, j] = np.where(
                valid, 1 - (abs_vals - mn) / (mx - mn), 0.5)
    elif key == 'max_dd':
        mn, mx = np.nanmin(col_vals[valid]), np.nanmax(col_vals[valid])
        if mx > mn:
            color_mat[:, j] = np.where(
                valid, (col_vals - mn) / (mx - mn), 0.5)
    else:
        mn, mx = np.nanmin(col_vals[valid]), np.nanmax(col_vals[valid])
        if mx > mn:
            color_mat[:, j] = np.where(
                valid,
                (col_vals - mn) / (mx - mn) if hib
                else 1 - (col_vals - mn) / (mx - mn),
                0.5)

cmap = LinearSegmentedColormap.from_list(
    'gwr',
    ['#C62828','#EF9A9A','#FFFFFF','#A5D6A7','#2E7D32'],
    N=256)

fig, ax = plt.subplots(figsize=(15, 5))
ax.axis('off')
fig.patch.set_facecolor('#FAFAFA')

name_w = 0.20
data_w = (1.0 - name_w) / n_cols
row_h  = 1.0 / (n_strats + 2)

ax.add_patch(plt.Rectangle(
    (0, 1-row_h), name_w, row_h,
    fc='#1A3A5C', ec='white', lw=0.8,
    transform=ax.transAxes, clip_on=False))
ax.text(name_w/2, 1-row_h/2, 'Strategy',
        ha='center', va='center', fontsize=9,
        fontweight='bold', color='white',
        transform=ax.transAxes)

for j, (_, col_name, _, _) in enumerate(columns):
    x = name_w + j*data_w
    ax.add_patch(plt.Rectangle(
        (x, 1-row_h), data_w, row_h,
        fc='#1A3A5C', ec='white', lw=0.8,
        transform=ax.transAxes, clip_on=False))
    ax.text(x+data_w/2, 1-row_h/2, col_name,
            ha='center', va='center', fontsize=8.5,
            fontweight='bold', color='white',
            transform=ax.transAxes)

for i, (strat, label) in enumerate(zip(strategies, labels)):
    y       = 1 - row_h*(i+2)
    name_bg = '#E8EAF6' if i%2==0 else '#F3F4F8'
    ax.add_patch(plt.Rectangle(
        (0, y), name_w, row_h,
        fc=name_bg, ec='#CCCCCC', lw=0.4,
        transform=ax.transAxes, clip_on=False))
    ax.text(0.008, y+row_h/2, label,
            ha='left', va='center', fontsize=8.5,
            fontweight='bold' if strat not in
            ('FF5_Linear','HML_Passive') else 'normal',
            color='#1A3A5C',
            transform=ax.transAxes)

    for j in range(n_cols):
        x    = name_w + j*data_w
        cval = color_mat[i, j]
        rgba = list(cmap(cval))
        if strat in ('FF5_Linear','HML_Passive'):
            rgba = [0.7*c+0.3 for c in rgba[:3]] + [0.75]

        ax.add_patch(plt.Rectangle(
            (x, y), data_w, row_h,
            fc=rgba, ec='#CCCCCC', lw=0.4,
            transform=ax.transAxes, clip_on=False))

        txt = txt_mat[i][j]
        fw  = 'bold' if j in (0,2,5,6) else 'normal'
        lum = 0.299*rgba[0] + 0.587*rgba[1] + 0.114*rgba[2]
        fc  = '#1A1A1A' if lum > 0.45 else 'white'
        ax.text(x+data_w/2, y+row_h/2, txt,
                ha='center', va='center', fontsize=8.5,
                fontweight=fw, color=fc,
                transform=ax.transAxes)

sm_ = plt.cm.ScalarMappable(
    cmap=cmap, norm=plt.Normalize(vmin=0,vmax=1))
sm_.set_array([])
cax = fig.add_axes([0.92, 0.15, 0.012, 0.65])
cb  = fig.colorbar(sm_, cax=cax)
cb.set_ticks([0,0.5,1])
cb.set_ticklabels(['Worst','Median','Best'], fontsize=7)

fig.text(0.5, 0.01,
    'Alpha & t-stat: HAC SE (6 lags) vs FF5. '
    'IR = active return / tracking error vs FF5 factor-explained return. '
    'HML Load & MKT Beta: closer to 0 = more market-neutral = greener. '
    'All returns net of 50bps TC, 150bps borrow, RF rebate, Reg T ÷1.5.',
    ha='center', fontsize=7, color='#777')

ax.set_title('Full OOS Performance (2015–2024, net of all costs)',
             fontsize=13, fontweight='bold', color='#1A3A5C', pad=10)

plt.savefig('slide13_heatmap_table.png', dpi=200,
            bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print("Saved: slide13_heatmap_table.png")

## Cell B — Equity curves and drawdown with IVE and VTV benchmarks

Adds IVE (iShares S&P 500 Value ETF) and VTV (Vanguard Value ETF) to the equity curve and drawdown chart. These are the practitioner-relevant value benchmarks. Their inclusion contextualises our ML strategy against what a real value investor could have done with passive ETFs over the same period.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import statsmodels.api as sm
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# Load the saved allocation CSV
master = pd.read_csv('master_allocation.csv', parse_dates=['date'])

def get_rets(strat):
    return (master[master['strategy']==strat]
            .set_index('date')['ls_ret']
            .sort_index())

rets = {s: get_rets(s) for s in master['strategy'].unique()}
idx  = rets['A_Value'].index

# Pull FF5 for RF series
import requests, zipfile, io
from io import StringIO

url = ("https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/"
       "ftp/F-F_Research_Data_5_Factors_2x3_CSV.zip")
r   = requests.get(url)
z   = zipfile.ZipFile(io.BytesIO(r.content))
with z.open(z.namelist()[0]) as f:
    raw = f.read().decode('utf-8')
lines = raw.split('\n')
start = 0
for i, line in enumerate(lines):
    s = line.strip()
    if s and s[0].isdigit() and len(s.split(',')[0].strip()) == 6:
        start = i
        break
ff5 = pd.read_csv(
    StringIO('\n'.join(lines[start:])),
    header=None,
    names=['date','MKT','SMB','HML','RMW','CMA','RF'],
    on_bad_lines='skip'
)
ff5 = ff5[ff5['date'].astype(str).str.strip().str.len()==6]
ff5['date'] = pd.to_datetime(
    ff5['date'].astype(str).str.strip(), format='%Y%m')
ff5 = ff5.set_index('date').apply(pd.to_numeric, errors='coerce') / 100
ff5 = ff5.dropna()
rf  = ff5['RF']

# Pull IVE and VTV from Yahoo Finance
# IVE = iShares S&P 500 Value (most cited practitioner value benchmark)
# VTV = Vanguard Value ETF (tracks CRSP US Large Cap Value — close to our methodology)
print("Pulling value ETF data from Yahoo Finance...")

etf_tickers = {'IVE': 'iShares S&P500 Value', 'VTV': 'Vanguard Value'}
etf_rets     = {}

for ticker, name in etf_tickers.items():
    raw_etf = yf.download(ticker,
                           start='2014-12-01',
                           end='2024-12-31',
                           progress=False,
                           auto_adjust=True)

    # Newer yfinance versions return MultiIndex columns — flatten if needed
    if isinstance(raw_etf.columns, pd.MultiIndex):
        raw_etf.columns = raw_etf.columns.get_level_values(0)

    price   = raw_etf['Close']
    monthly = price.resample('MS').last()
    ret_etf = monthly.pct_change().dropna()
    ret_etf = ret_etf.reindex(idx).dropna()
    etf_rets[ticker] = ret_etf
    print(f"  {ticker}: {len(ret_etf)} months, "
          f"{ret_etf.index.min().strftime('%Y-%m')} to "
          f"{ret_etf.index.max().strftime('%Y-%m')}")

ive = etf_rets['IVE']
vtv = etf_rets['VTV']

# Metric helpers for the IVE alpha table
def ann_ret(r):
    r = r.dropna()
    return (1+r).prod() ** (12/len(r)) - 1

def ann_vol(r):
    return r.dropna().std() * np.sqrt(12)

def sharpe(r, rf_s):
    r = r.dropna()
    ex = r - rf_s.reindex(r.index).fillna(0)
    return ex.mean() / ex.std() * np.sqrt(12)

def maxdd(r):
    r = r.dropna()
    cum = (1+r).cumprod()
    return ((cum - cum.cummax()) / cum.cummax()).min()

def alpha_vs_benchmark(r, bench, rf_s):
    # Jensen alpha: regress (strategy - RF) on (benchmark - RF)
    # Beta close to zero confirms market-neutrality vs the benchmark
    r = r.dropna()
    b = bench.reindex(r.index).dropna()
    r = r.reindex(b.index)
    rf_a = rf_s.reindex(r.index).fillna(0)
    y    = r  - rf_a
    X    = b  - rf_a
    X    = sm.add_constant(X)
    res  = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags':6})
    return res.params['const']*12, res.tvalues['const'], res.params.iloc[1]

def ff5_alpha(r):
    r = r.dropna()
    X = ff5[['MKT','SMB','HML','RMW','CMA']].reindex(r.index).dropna()
    r = r.reindex(X.index)
    res = sm.OLS(r, sm.add_constant(X)).fit(
        cov_type='HAC', cov_kwds={'maxlags':6})
    return res.params['const']*12, res.tvalues['const']

stars_fn = lambda t: ('***' if abs(t)>3 else '**' if abs(t)>2
                       else '*' if abs(t)>1.65 else '') \
                      if not np.isnan(t) else ''

# Print IVE alpha for each ML strategy
print("\n" + "="*60)
print("ALPHA vs IVE — HAC SE (6 lags)")
print("="*60)
ml_strats = ['A_Value','B_ValueQuality','C_ValueQualityMom','D_ValMOM']
for strat in ml_strats:
    r = rets.get(strat, pd.Series(dtype=float))
    if len(r) == 0: continue
    a_ive, t_ive, b_ive = alpha_vs_benchmark(r, ive, rf)
    a_ff5, t_ff5        = ff5_alpha(r)
    print(f"\n{strat}:")
    print(f"  Alpha vs IVE: {a_ive:.2%}/yr  (t={t_ive:.2f}{stars_fn(t_ive)})  "
          f"beta={b_ive:.3f}")
    print(f"  Alpha vs FF5: {a_ff5:.2%}/yr  (t={t_ff5:.2f}{stars_fn(t_ff5)})")

print(f"\nIVE: Ann Return={ann_ret(ive):.2%}  Sharpe={sharpe(ive,rf):.2f}  "
      f"Max DD={maxdd(ive):.2%}")
print(f"VTV: Ann Return={ann_ret(vtv):.2%}  Sharpe={sharpe(vtv,rf):.2f}  "
      f"Max DD={maxdd(vtv):.2%}")

# Equity curves and drawdown with ETFs
colors_map = {
    'A_Value':           '#9E9E9E',
    'B_ValueQuality':    '#43A047',
    'C_ValueQualityMom': '#1E88E5',
    'D_ValMOM':          '#FF8F00',
    'FF5_Linear':        '#8E24AA',
    'HML_Passive':       '#E53935',
    'IVE':               '#00838F',
    'VTV':               '#004D40',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#FAFAFA')
fig.suptitle('ML Value Strategy vs Value ETF Benchmark (2015–2024)',
             fontsize=13, fontweight='bold', color='#1A3A5C', y=1.01)

ax = axes[0]
for strat, r in rets.items():
    nav = (1+r).cumprod()
    ls  = '--' if strat == 'HML_Passive' else \
          ':' if strat == 'FF5_Linear' else '-'
    lw  = 2.5 if strat == 'D_ValMOM' else 1.5
    ax.semilogy(nav.index, nav.values,
                color=colors_map[strat], lw=lw, ls=ls,
                label=strat.replace('_',' '), alpha=0.85)

ive_nav = (1+ive).cumprod()
vtv_nav = (1+vtv).cumprod()
ax.semilogy(ive_nav.index, ive_nav.values,
            color=colors_map['IVE'], lw=2.2, ls='-.',
            label='IVE (S&P500 Value ETF)', alpha=0.95)
ax.semilogy(vtv_nav.index, vtv_nav.values,
            color=colors_map['VTV'], lw=1.8, ls='-.',
            label='VTV (Vanguard Value ETF)', alpha=0.8)

ax.axvspan(pd.Timestamp('2015-01-01'), pd.Timestamp('2020-12-01'),
           alpha=0.06, color='#E53935')
ax.text(pd.Timestamp('2017-06-01'), 0.82, 'Value Lost Decade',
        fontsize=8, color='#C62828', alpha=0.7, ha='center', style='italic')
ax.axvline(pd.Timestamp('2022-01-01'), color='#37474F', lw=1, ls=':', alpha=0.5)
ax.set_title('Equity Curves ($1 investi, échelle log)',
             fontweight='bold', color='#1A3A5C')
ax.set_ylabel('NAV (log scale)')
ax.legend(fontsize=7, framealpha=0.9, ncol=2)
ax.grid(alpha=0.15)
ax.set_facecolor('#FAFAFA')
for s in ['top','right']: ax.spines[s].set_visible(False)

ax = axes[1]
for strat, r in rets.items():
    cum = (1+r).cumprod()
    dd  = (cum - cum.cummax()) / cum.cummax()
    ls  = '--' if strat == 'HML_Passive' else \
          ':' if strat == 'FF5_Linear' else '-'
    ax.fill_between(dd.index, dd.values, 0, alpha=0.08, color=colors_map[strat])
    ax.plot(dd.index, dd.values, color=colors_map[strat], lw=1.5, ls=ls,
            label=strat.replace('_',' '), alpha=0.85)

for ticker, r_etf, color in [('IVE', ive, colors_map['IVE']),
                               ('VTV', vtv, colors_map['VTV'])]:
    cum = (1+r_etf).cumprod()
    dd  = (cum - cum.cummax()) / cum.cummax()
    ax.fill_between(dd.index, dd.values, 0, alpha=0.12, color=color)
    ax.plot(dd.index, dd.values, color=color, lw=2.2, ls='-.', label=ticker, alpha=0.95)

ax.axvspan(pd.Timestamp('2015-01-01'), pd.Timestamp('2020-12-01'),
           alpha=0.06, color='#E53935')
ax.set_title('Drawdown comparatif', fontweight='bold', color='#1A3A5C')
ax.set_ylabel('Drawdown')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{x:.0%}"))
ax.legend(fontsize=7, framealpha=0.9, ncol=2)
ax.grid(alpha=0.15)
ax.set_facecolor('#FAFAFA')
for s in ['top','right']: ax.spines[s].set_visible(False)

plt.tight_layout()
plt.savefig('slide14_with_ive.png', dpi=200, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print("Saved: slide14_with_ive.png")

## Cell C — Short book market cap diagnostic

This diagnostic was run to understand the liquidity profile of the short book — specifically to answer the question: are the stocks we are shorting small enough that borrow costs of 150bps/yr are justified?

We approximate the short book as the bottom decile of actual returns each month (a proxy for the bottom decile of predicted returns) and look at the market equity distribution by decile. The results confirm that the bottom decile has a median market cap of ~$59M — solidly small cap — justifying our borrow cost assumption.

In [ ]:
import pandas as pd
import numpy as np

# Load market equity data from both datasets
kelly = pd.read_csv('kelly_final.csv',
                     usecols=['permno','date','ret','me'],
                     parse_dates=['date'])

# Only use OOS period — the short book we are diagnosing is the OOS short book
oos = kelly[kelly['date'] >= '2015-01-01']

# Proxy for the short book: bottom decile of actual returns each month
# This approximates what the model selects since predicted and actual returns
# are correlated (IC ~0.08)
oos_monthly = oos.groupby('date').apply(
    lambda x: x.assign(ret_decile=pd.qcut(
        x['ret'].rank(pct=True), 10,
        labels=False, duplicates='drop'))
).reset_index(drop=True)

me_by_decile = oos_monthly.groupby('ret_decile')['me'].describe()
print("Market equity par décile (millions USD):")
print(me_by_decile[['mean','50%','25%','75%']].round(0))

## Cell D — AUM capacity analysis (preliminary version)

An earlier version of the AUM capacity calculation, produced to get a quick estimate before the full chart in Cell E. Uses a simple estimated ADV function based on market cap buckets rather than real CRSP daily volume data.

In [ ]:
import pandas as pd
import numpy as np

# Load both datasets to cover the full OOS period including extension
master = pd.read_csv('master_allocation.csv', parse_dates=['date'])

print("Loading market equity data...")
kelly = pd.read_csv('kelly_final.csv',
                     usecols=['permno','date','ret','me'],
                     parse_dates=['date'])

ext = pd.read_csv('features_2022_2024.csv',
                   usecols=['permno','date','ret','me'],
                   parse_dates=['date'])

all_data = pd.concat([kelly, ext], ignore_index=True)
all_data = all_data.sort_values(['permno','date'])

oos = all_data[all_data['date'] >= '2015-01-01'].copy()

def estimate_adv(me_millions):
    """
    Estimate ADV from market cap using empirical bucket ratios.
    Small caps trade more volume relative to size (higher float turnover)
    but in dollar terms much less than large caps.
    """
    if me_millions < 100:
        return me_millions * 0.004   # micro cap: ~0.4% of market cap per day
    elif me_millions < 500:
        return me_millions * 0.005   # small cap: ~0.5%
    elif me_millions < 2000:
        return me_millions * 0.006   # mid cap: ~0.6%
    else:
        return me_millions * 0.007   # large cap: ~0.7%

oos['adv_est'] = oos['me'].apply(estimate_adv)

monthly_adv = oos.groupby('date')['adv_est'].describe()
print("\nMonthly ADV distribution (millions USD):")
print(monthly_adv[['mean','25%','50%','75%']].tail(12).round(2))

def compute_aum_capacity(
    oos_data,
    n_short=500,        # ~500 stocks in the short book
    adv_limit=0.10,     # max 10% of daily volume per position per day
    trading_days=21,    # spread position build over one full month
):
    """
    Computes the maximum equity sleeve AUM such that each short position
    can be built within the ADV limit spread over trading_days.

    Logic:
      Max daily trade per stock = adv_limit × ADV
      Max position per stock    = max daily trade × trading_days
      Max equity AUM            = max position × n_short / 0.5
        (0.5 because short book = 50% of equity sleeve NAV)
    """
    results = []
    for date, month in oos_data.groupby('date'):
        if len(month) < 50:
            continue
        # Use the 25th percentile of the bottom quartile as the binding constraint
        # This is more conservative than the median and closer to reality
        month_sorted    = month.sort_values('me')
        n_bottom        = len(month_sorted) // 4
        bottom          = month_sorted.head(n_bottom)
        avg_adv_binding = bottom['adv_est'].quantile(0.25)
        max_pos         = avg_adv_binding * adv_limit * trading_days
        aum_capacity    = max_pos * n_short / 0.5
        results.append({'date': date, 'aum_capacity_M': aum_capacity})

    return pd.DataFrame(results).set_index('date')

print("\nRunning AUM capacity analysis...")
capacity = compute_aum_capacity(oos)

print("\nAUM Capacity over time (millions USD):")
print(f"  Median capacity:  ${capacity['aum_capacity_M'].median():.0f}M")
print(f"  Mean capacity:    ${capacity['aum_capacity_M'].mean():.0f}M")
print(f"  25th percentile:  ${capacity['aum_capacity_M'].quantile(0.25):.0f}M")
print(f"  10th percentile:  ${capacity['aum_capacity_M'].quantile(0.10):.0f}M")
print(f"  Min capacity:     ${capacity['aum_capacity_M'].min():.0f}M")

## Cell E — Full AUM capacity chart (Slide 17)

The complete 6-panel AUM capacity analysis that produces the chart for Slide 17 of the presentation. Contains the 4-step logic diagram, sensitivity heatmap, capacity over time, and the revenue model at our $200M target AUM.

Key result: equity sleeve capacity of $126M (conservative, anchored on 10th percentile of bottom quartile) to $291M (base case, anchored on 25th percentile). Total fund AUM at 50% equity allocation: $252M to $582M. We target $200M total AUM — well within the conservative ceiling — to preserve alpha quality.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# AUM CAPACITY ANALYSIS — SLIDE 17
#
# The capacity ceiling is determined by the short book:
# our short book contains small-cap value traps with limited daily volume.
# If the fund is too large, building short positions moves prices against us.
#
# 4-step logic:
#   1. Median short book market cap → estimated ADV (0.5% × market cap)
#   2. Apply 10% ADV rule (institutional standard)
#   3. Spread position build over 21 trading days
#   4. With 500 short positions: AUM capacity = max position × 500 / 0.5
# ============================================================

print("=" * 65)
print("AUM CAPACITY ANALYSIS — ML Value Long-Short Strategy")
print("=" * 65)

print("\nLoading data...")
kelly = pd.read_csv('kelly_final.csv',
                    usecols=['permno', 'date', 'ret', 'me'],
                    parse_dates=['date'])
ext   = pd.read_csv('features_2022_2024.csv',
                    usecols=['permno', 'date', 'ret', 'me'],
                    parse_dates=['date'])

all_data = pd.concat([kelly, ext], ignore_index=True)
oos      = all_data[all_data['date'] >= '2015-01-01'].copy()
oos      = oos[oos['me'].notna() & (oos['me'] > 0)]

print(f"  OOS period: {oos['date'].min().strftime('%Y-%m')} "
      f"to {oos['date'].max().strftime('%Y-%m')}")
print(f"  Total stock-months: {len(oos):,}")

# Step 1: short book market cap distribution
monthly_stats = []
for date, month in oos.groupby('date'):
    if len(month) < 50: continue
    sorted_month = month.sort_values('me')
    n_bottom     = len(sorted_month) // 4
    bottom       = sorted_month.head(n_bottom)
    monthly_stats.append({
        'date':    date,
        'p10_me':  bottom['me'].quantile(0.10),
        'p25_me':  bottom['me'].quantile(0.25),
        'p50_me':  bottom['me'].quantile(0.50),
        'p75_me':  bottom['me'].quantile(0.75),
        'mean_me': bottom['me'].mean(),
    })
stats_df = pd.DataFrame(monthly_stats).set_index('date')

p10 = stats_df['p10_me'].median()
p25 = stats_df['p25_me'].median()
p50 = stats_df['p50_me'].median()
p75 = stats_df['p75_me'].median()

print(f"\nShort book candidates — market cap distribution:")
print(f"  10th percentile (most illiquid):  ${p10:.0f}M")
print(f"  25th percentile:                  ${p25:.0f}M")
print(f"  Median:                           ${p50:.0f}M")
print(f"  75th percentile:                  ${p75:.0f}M")

# Step 2: estimated daily volume (ADV = 0.5% × market cap)
adv_pct = 0.005
adv_p10 = p10 * adv_pct
adv_p25 = p25 * adv_pct
adv_p50 = p50 * adv_pct

# Step 3: max position per stock (10% ADV × 21 days)
adv_limit      = 0.10
execution_days = 21
n_short        = 500
short_weight   = 0.50

max_pos_p10 = adv_p10 * adv_limit * execution_days
max_pos_p25 = adv_p25 * adv_limit * execution_days
max_pos_p50 = adv_p50 * adv_limit * execution_days

# Step 4: AUM capacity
aum_conservative = max_pos_p10 * n_short / short_weight
aum_base         = max_pos_p25 * n_short / short_weight
aum_generous     = max_pos_p50 * n_short / short_weight

fund_conservative = aum_conservative / 0.50
fund_base         = aum_base         / 0.50
fund_generous     = aum_generous     / 0.50

print(f"\nEquity sleeve capacity:")
print(f"  Conservative (10th pctile):  ${aum_conservative:.0f}M")
print(f"  Base case    (25th pctile):  ${aum_base:.0f}M")
print(f"  Generous     (median):       ${aum_generous:.0f}M")
print(f"\nTotal fund AUM (equity = 50%):")
print(f"  Conservative:  ${fund_conservative:.0f}M")
print(f"  Base case:     ${fund_base:.0f}M")
print(f"  Generous:      ${fund_generous:.0f}M")

# Monthly capacity over time
monthly_capacity = []
for date, month in oos.groupby('date'):
    if len(month) < 50: continue
    sorted_month = month.sort_values('me')
    n_bottom     = len(sorted_month) // 4
    bottom       = sorted_month.head(n_bottom)
    me_binding   = bottom['me'].quantile(0.25)
    adv_binding  = me_binding * adv_pct
    max_pos      = adv_binding * adv_limit * execution_days
    aum_eq       = max_pos * n_short / short_weight
    monthly_capacity.append({
        'date':       date,
        'me_binding': me_binding,
        'aum_equity': aum_eq,
        'aum_fund':   aum_eq / 0.50,
    })
cap_df = pd.DataFrame(monthly_capacity).set_index('date')

# Revenue model at $200M target
target_total_aum = 200
target_eq_aum    = target_total_aum * 0.50
net_return       = 0.1823

mgmt_fee  = target_total_aum * 0.02
perf_fee  = target_total_aum * net_return * 0.20
total_rev = mgmt_fee + perf_fee

print(f"\nRevenue at ${target_total_aum}M total AUM (2/20 fees):")
print(f"  Management fee (2%):   ${mgmt_fee:.1f}M/year")
print(f"  Performance fee (20%): ${perf_fee:.1f}M/year")
print(f"  Total:                 ${total_rev:.1f}M/year")

# Sensitivity table
print(f"\n{'':30} {'5% ADV':>10} {'10% ADV':>10} {'15% ADV':>12}")
print("-"*65)
for td in [10, 21, 42]:
    row = f"  Build over {td:>2} trading days       "
    for adv_lim in [0.05, 0.10, 0.15]:
        cap = p25 * adv_pct * adv_lim * td * n_short / short_weight
        row += f"  ${cap:>5.0f}M"
    print(row)

# ── 6-panel chart ──────────────────────────────────────────
BLUE  = '#1565C0'
GREEN = '#2E7D32'
RED   = '#C62828'
GOLD  = '#F57F17'
MGREY = '#E0E0E0'
LGREY = '#F5F5F5'

fig = plt.figure(figsize=(15, 11))
fig.patch.set_facecolor('#FAFAFA')
gs  = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Panel 1: market cap distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.fill_between(stats_df.index, stats_df['p10_me'], stats_df['p75_me'],
                 alpha=0.15, color=BLUE, label='10th-75th pctile')
ax1.plot(stats_df['p50_me'], color=BLUE, lw=2,
         label=f'Median (${stats_df["p50_me"].median():.0f}M)')
ax1.plot(stats_df['p25_me'], color=RED, lw=1.5, ls='--',
         label=f'25th pctile (${stats_df["p25_me"].median():.0f}M)')
ax1.axhline(100, color=GOLD, lw=1, ls=':', alpha=0.8, label='$100M reference')
ax1.set_title('Short Book\nMarket Cap Distribution',
              fontweight='bold', color='#1A3A5C', fontsize=10)
ax1.set_ylabel('Market Cap ($M)')
ax1.legend(fontsize=7, framealpha=0.9)
ax1.grid(alpha=0.15)
ax1.set_facecolor('#FAFAFA')
for s in ['top','right']: ax1.spines[s].set_visible(False)

# Panel 2: ADV estimates
ax2 = fig.add_subplot(gs[0, 1])
adv_median = stats_df['p25_me'] * adv_pct * 1000
ax2.plot(adv_median, color=GREEN, lw=2,
         label=f'25th pctile (${adv_median.median():.0f}K/day)')
ax2.fill_between(adv_median.index,
                 stats_df['p10_me'] * adv_pct * 1000,
                 stats_df['p50_me'] * adv_pct * 1000,
                 alpha=0.15, color=GREEN)
ax2.axhline(adv_p25*1000, color=RED, lw=1.5, ls='--',
            label=f'Overall median: ${adv_p25*1000:.0f}K/day')
ax2.set_title('Estimated Daily Volume\n(ADV = 0.5% × Market Cap)',
              fontweight='bold', color='#1A3A5C', fontsize=10)
ax2.set_ylabel('ADV ($K/day)')
ax2.legend(fontsize=7, framealpha=0.9)
ax2.grid(alpha=0.15)
ax2.set_facecolor('#FAFAFA')
for s in ['top','right']: ax2.spines[s].set_visible(False)

# Panel 3: AUM capacity over time
ax3 = fig.add_subplot(gs[0, 2])
ax3.fill_between(cap_df.index,
                 cap_df['aum_equity'] * 0.6, cap_df['aum_equity'] * 1.4,
                 alpha=0.1, color=BLUE, label='±40% range')
ax3.plot(cap_df['aum_equity'], color=BLUE, lw=2,
         label=f"Base case (median ${cap_df['aum_equity'].median():.0f}M)")
ax3.axhline(aum_conservative, color=RED, lw=2, ls='--',
            label=f'Conservative ${aum_conservative:.0f}M')
ax3.axhline(target_eq_aum, color=GREEN, lw=2, ls=':',
            label=f'Our target ${target_eq_aum:.0f}M')
ax3.set_title('Equity Sleeve\nAUM Capacity Over Time',
              fontweight='bold', color='#1A3A5C', fontsize=10)
ax3.set_ylabel('Max Equity AUM ($M)')
ax3.legend(fontsize=7, framealpha=0.9)
ax3.grid(alpha=0.15)
ax3.set_facecolor('#FAFAFA')
for s in ['top','right']: ax3.spines[s].set_visible(False)

# Panel 4: 4-step logic diagram
ax4 = fig.add_subplot(gs[1, 0])
ax4.axis('off')
steps = [
    ("STEP 1", f"Short book\nMedian cap: ${p50:.0f}M\n25th pctile: ${p25:.0f}M",
     '#E3F2FD', BLUE),
    ("STEP 2", f"Daily volume (ADV)\n0.5% × market cap\n${adv_p25*1000:.0f}K/day",
     '#E8F5E9', GREEN),
    ("STEP 3", f"Max position\n10% ADV × 21 days\n${max_pos_p25*1000:.0f}K/stock",
     '#FFF3E0', GOLD),
    ("STEP 4", f"AUM capacity\n${max_pos_p25*1000:.0f}K × 500 stocks ÷ 50%\n"
               f"= ${aum_base:.0f}M equity sleeve",
     '#FCE4EC', RED),
]
for i, (label, text, bg, col) in enumerate(steps):
    y    = 0.75 - i * 0.22
    rect = mpatches.FancyBboxPatch(
        (0.05, y-0.08), 0.90, 0.18,
        boxstyle="round,pad=0.02",
        facecolor=bg, edgecolor=col, linewidth=1.5,
        transform=ax4.transAxes, clip_on=False)
    ax4.add_patch(rect)
    ax4.text(0.12, y+0.02, label, fontsize=8, fontweight='bold', color=col,
             transform=ax4.transAxes, va='center')
    ax4.text(0.45, y+0.01, text, fontsize=7.5, color='#1A1A1A',
             transform=ax4.transAxes, va='center', ha='center')
    if i < 3:
        ax4.annotate('', xy=(0.5, y-0.08), xytext=(0.5, y-0.12),
                     xycoords='axes fraction',
                     arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
ax4.set_title('The 4-Step Capacity Logic',
              fontweight='bold', color='#1A3A5C', fontsize=10, pad=10)

# Panel 5: sensitivity heatmap
ax5 = fig.add_subplot(gs[1, 1])
adv_limits  = [0.05, 0.10, 0.15, 0.20]
exec_days   = [10, 21, 42]
sens_matrix = np.array([[p25 * adv_pct * a * d * n_short / short_weight
                          for a in adv_limits]
                         for d in exec_days])

im = ax5.imshow(sens_matrix, cmap='RdYlGn', vmin=0, vmax=500, aspect='auto')
ax5.set_xticks(range(len(adv_limits)))
ax5.set_xticklabels([f'{a:.0%}' for a in adv_limits], fontsize=8)
ax5.set_yticks(range(len(exec_days)))
ax5.set_yticklabels([f'{d}d' for d in exec_days], fontsize=8)
ax5.set_xlabel('ADV limit per day', fontsize=8)
ax5.set_ylabel('Execution days', fontsize=8)
for i in range(len(exec_days)):
    for j in range(len(adv_limits)):
        val = sens_matrix[i, j]
        col = 'white' if val < 150 or val > 400 else '#1A1A1A'
        ax5.text(j, i, f'${val:.0f}M', ha='center', va='center',
                 fontsize=8, fontweight='bold', color=col)
ax5.set_title('Sensitivity: Equity Sleeve\nCapacity ($M)',
              fontweight='bold', color='#1A3A5C', fontsize=10)
plt.colorbar(im, ax=ax5, shrink=0.8, label='Equity AUM ($M)')

# Panel 6: revenue model
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
rev_scenarios = [
    ("Conservative\n$150M total AUM",  150,  0.1070),
    ("Base case\n$200M total AUM",      200,  0.1823),
    ("Optimistic\n$300M total AUM",     300,  0.1500),
]
y_pos = 0.85
ax6.text(0.5, 0.97, 'Revenue Model (2% mgmt + 20% perf)',
         ha='center', fontsize=9, fontweight='bold',
         color='#1A3A5C', transform=ax6.transAxes)
for label, aum, ret in rev_scenarios:
    mf = aum * 0.02
    pf = aum * ret * 0.20
    tr = mf + pf
    col = GREEN if aum == 200 else '#555'
    fw  = 'bold' if aum == 200 else 'normal'
    ax6.text(0.05, y_pos, label, fontsize=8, color=col, fontweight=fw,
             transform=ax6.transAxes)
    ax6.text(0.55, y_pos,
             f"Mgmt: ${mf:.1f}M\nPerf: ${pf:.1f}M\nTotal: ${tr:.1f}M/yr",
             fontsize=7.5, color=col, fontweight=fw, transform=ax6.transAxes)
    y_pos -= 0.27

ax6.axhline(0.05, color=MGREY, lw=0.5)
ax6.text(0.5, 0.02,
         '★ Base case at $200M delivers ~$11M/yr revenue',
         ha='center', fontsize=8, color=GREEN, fontweight='bold',
         transform=ax6.transAxes, style='italic')

fig.suptitle(
    'AUM Capacity Analysis — ML Value L/S Equity Strategy\n'
    f'Equity sleeve: ${aum_conservative:.0f}M (conservative) '
    f'to ${aum_base:.0f}M (base) | '
    f'Total fund target: $200M',
    fontsize=12, fontweight='bold', color='#1A3A5C', y=1.01)

plt.savefig('aum_capacity_full.png', dpi=180,
            bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print("\nSaved: aum_capacity_full.png")